对于最佳的前五类模型（xgboost，adaboost，rf，gboost,knn）进行进一步优化，考虑到固定网格搜索选择的参数比较有限，在更小范围内进行参数搜索，结果保存在scores_with_best_parameters中

In [1]:
from sklearn.model_selection import GridSearchCV

# 这里采用xgboost算法进行预测
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold,train_test_split,cross_validate,cross_val_score,KFold
from sklearn.metrics import r2_score, mean_absolute_percentage_error,mean_absolute_error,make_scorer
from xgboost import XGBRegressor
from utils import create_empty_ls,create_dir,mycopyfile,mycopydir
import joblib
from openpyxl import load_workbook
import os
"""
路径的配置和选择
"""
model_name='xgboost'

save_model=True
del_0qf=True #是否删除0屈服强度数据
main_path=f'models/{model_name}' # 这里是模型所在的文件位置
create_dir(main_path,is_mainpath=True)
# excel='index/xgboost_index.xlsx'
# 如果分数表格不存在就新建一个
score_path='scores_with_best_parameters.xlsx'
if os.path.exists(score_path):
    print(f"The file '{score_path}' exists.")
else:
    print(f"The file '{score_path}' does not exist.")
    df = pd.DataFrame(columns=['Model', 'R^2', 'MAPE', 'MAE'])
    df.to_excel(score_path, index=False)


"""
文件读取部分
"""
# 读取 Excel 文件
df = pd.read_excel('train_set_new.xlsx', index_col=0)  # 引入这一列之后，原本的第一列就是一个索引，第0列会从有意义的列开始

display(df)
np.random.seed(200) #设置随机数种子
feature_names = df.drop(columns=['Precipitate Distribution'
       ,'屈服强度', '抗拉强度 (UTS)', '追踪编号']).columns
X = df.drop(columns=['Precipitate Distribution'
       ,'屈服强度', '抗拉强度 (UTS)', '追踪编号'])  # 特征: 最后四列之前的所有列
y = df.iloc[:, -3]  # 目标: 倒数第四列
# 将目标变量分箱为10个类别
bins = np.linspace(y.min(), y.max(), 11)
y_binned = np.digitize(y, bins)
# print('y_binned',y_binned)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2,random_state=200)

"""
设置初始化
"""
n_splits=5
# 初始化StratifiedKFold
kf = KFold(n_splits=5, shuffle=True, random_state=200)# 注意这里修改了random_state以确保每次初始化不同

# 初始化XGBoost回归器
model = XGBRegressor(random_state=200)# 设置随机种子以确保结果的可重复性
# 初始化列表以存储每个折叠的分数
# 定义评估指标
scoring = {
    'r2':make_scorer(r2_score,greater_is_better=True),
    'MAE': make_scorer(mean_absolute_error, greater_is_better=False),
    'MAPE': make_scorer(mean_absolute_percentage_error, greater_is_better=False)
}
r2_scores_ls = []
mape_scores = []
mae_scores=[]
best_r2_score = float('-inf')
best_mae_score= float('-inf')
best_mape_score= float('-inf')
ls1=['fold'+str(i) for i in range(1,n_splits+1)]
fold_dict=dict(zip(ls1,[0 for i in range(0,len(ls1))]))

# 可修改
parameters={'max_depth':[n for n in range(1,7,2)],#1,3,5
            'n_estimators':[50*n for n in range(1,5)],#50,100,150,200
            'learning_rate':[0.01,0.1,0.2], #0.01,0.1,0.2
            'gamma':[0.1,0.2]} #

# 可修改
model = XGBRegressor(random_state=200)  # 注意这里修改了random_state以确保每次初始化不同
grid_search=GridSearchCV(model,parameters,scoring='r2',cv=kf) # 选择r2作为性能评估参数
grid_search.fit(X_train,y_train)
best_parameter=grid_search.best_estimator_
best_parameter

# 可修改
model = XGBRegressor(gamma=best_parameter.gamma,learning_rate=best_parameter.learning_rate,max_depth=best_parameter.max_depth)  # 注意这里修改了random_state以确保每次初始化不同
model.fit(X_train, y_train)
# 计算分折交叉验证结果
results = cross_validate(model,X,y,cv=kf, scoring=scoring, return_train_score=False)
# 输出分折交叉验证结果
for i, (r2, mae, mape) in enumerate(zip(results['test_r2'],results['test_MAE'], results['test_MAPE']), start=1):
    # 记录不同折的分数结果
    r2_scores_ls.append(r2)
    mape_scores.append(-mape)
    mae_scores.append(-mae)
    # print(f"折 {i}: r2={r2:.3f},MAE = {-mae:.3f}, MAPE = {-mape:.3f}")

current_r2_score=np.mean(r2_scores_ls)
# scores = cross_val_score(model, X, y_binned, cv=skf, scoring=make_scorer(r2_score))
# print('r2_scores:',scores)
# current_r2_score=scores.mean()
# 只对更优的结果进行保存
# 记录模型的特征重要性
feature_importance_final=model.feature_importances_
importance_dict = dict(zip(feature_importance_final, feature_names))
importance_tuplelist = [(abs(value), feature) for (value, feature) in importance_dict.items()]
importance_sorted = sorted(importance_tuplelist, reverse=True)
x_feature = [j for (i, j) in importance_sorted]
y_importance_value = [i for (i, j) in importance_sorted]
print(f'{model_name}_tree_feature_names(top15):')
for i in x_feature[:15]:
    print(i)
print(f'average_{model_name}_feature_values(top15):')
for i in y_importance_value[:15]:
    print(i)

#记录更优的分数
best_r2_score=current_r2_score
best_mae_score=np.mean(mae_scores)
best_mape_score=np.mean(mape_scores)
if save_model:
    joblib.dump(model,   f'models/{model_name}/{model_name}_optimization1.pkl')  #保存模型
    # print(f'保存更优{model_name}模型文件') 
print('平均 R2 分数:', best_r2_score)
print('平均 MAPE 分数:', best_mape_score * 100)  # 转换为百分比
print('平均MAE分数:',best_mae_score)    
"""
打印并输出分数至表格
"""

df = pd.read_excel(score_path)
# 添加新数据的示例
new_data = {'Model': f'{model_name}', 'R^2': best_r2_score, 'MAPE': best_mape_score * 100, 'MAE': best_mae_score}
new_data_df = pd.DataFrame([new_data])

# 使用 pd.concat() 合并原 DataFrame 和新的 DataFrame
df = pd.concat([df, new_data_df], ignore_index=True)
# 保存数据框到Excel表格
df.to_excel(score_path, index = False)

    

文件夹models/xgboost已存在
The file 'scores_with_best_parameters.xlsx' exists.


,MagpieData minimum Number,MagpieData maximum Number,MagpieData range Number,MagpieData mean Number,MagpieData avg_dev Number,MagpieData mode Number,MagpieData minimum MendeleevNumber,MagpieData maximum MendeleevNumber,MagpieData range MendeleevNumber,MagpieData mean MendeleevNumber,...,frac f valence electrons,Grain Size,Precipitate Distribution,Habit Plane,Calculated Solid Solution,Calculated Grain Boundary,Calculated Yield Strength,屈服强度,抗拉强度 (UTS),追踪编号
0,12,30,18,12.131276,0.247009,12,52,73,21,68.266826,...,0.000000,4.00,0,0,17.669989,194.321983,226.991971,364.0,393.0,No630-6Al-1Zn-0.15Mn -3
1,12,64,52,12.922815,1.811506,12,23,68,45,67.259700,...,0.054984,36.00,3,1,57.602249,48.504397,121.106646,125.0,200.0,No232-Mg-9Gd-1Sm-0.5Zr-2
2,12,64,52,13.780587,3.401587,12,12,69,57,66.376359,...,0.078055,50.00,3,5,94.613373,69.789053,179.402426,200.0,340.0,No16-Mg–14Gd–3Y–1.8Zn–0.5Zr -9
3,12,64,52,13.179217,2.292508,12,12,68,56,66.814372,...,0.052520,1.50,2,1,76.138523,262.520047,353.658570,290.0,280.0,No96-Mg-9.3Gd-2.7Y-0.65Ag-0.52Zr-0.18Ce -4
5,12,30,18,12.112392,0.217496,12,52,73,21,68.118900,...,0.000000,15.00,0,0,11.604694,86.484728,113.089422,148.0,280.0,No460-3Al-1Zn-0.3Mn -22
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
754,12,38,26,12.053936,0.104733,12,8,73,65,68.099036,...,0.000000,130.00,1,1,12.020723,23.509472,50.530195,58.5,220.0,No744-Mg-3Al-0.4Mn-0.05Sr
755,12,64,52,12.937303,1.825661,12,12,69,57,67.051202,...,0.033095,41.77,3,5,61.765688,53.715298,130.480986,223.0,295.0,No226-Mg-6Gd-3Y-1Zn-0.4Zr-0.7Ag-3
756,12,30,18,12.152650,0.281635,12,7,73,66,67.973850,...,0.000000,30.00,3,2,27.156102,71.029741,113.185842,91.1,144.3,No312-Mg-7.6Al-0.5Zn-1Ca-2
757,12,64,52,12.126080,0.237903,12,12,73,61,68.188783,...,0.003189,52.10,1,1,23.809563,45.036179,83.845741,125.0,217.0,No383-Mg-6Al-Zn-0.3Y-0.6Gd


xgboost_tree_feature_names(top15):
Interant electrons
Shear modulus local mismatch
frac f valence electrons
MagpieData mean MeltingT
Calculated Yield Strength
Mg fraction
MagpieData range MendeleevNumber
MagpieData maximum Column
MagpieData avg_dev AtomicWeight
Mn fraction
MagpieData range GSvolume_pa
MagpieData avg_dev Electronegativity
MagpieData avg_dev MeltingT
Grain Size
Lambda entropy
average_xgboost_feature_values(top15):
0.117761165
0.09272649
0.06941708
0.04193672
0.04095406
0.036480818
0.03246285
0.030818226
0.02780876
0.026451964
0.02305329
0.021004325
0.018452404
0.018379834
0.015192817
平均 R2 分数: 0.705730224181619
平均 MAPE 分数: 21.178651788533767
平均MAE分数: 32.65530757204236


In [2]:
print(X.shape)
print(best_parameter)

(739, 277)
XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=None, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             gamma=0.2, grow_policy=None, importance_type=None,
             interaction_constraints=None, learning_rate=0.1, max_bin=None,
             max_cat_threshold=None, max_cat_to_onehot=None,
             max_delta_step=None, max_depth=5, max_leaves=None,
             min_child_weight=None, missing=nan, monotone_constraints=None,
             multi_strategy=None, n_estimators=100, n_jobs=None,
             num_parallel_tree=None, random_state=200, ...)


In [3]:
# 看看随便更新下的参数结果
model = XGBRegressor(random_state=200)  # 注意这里修改了random_state以确保每次初始化不同
model.fit(X_train, y_train)
joblib.dump(model,   f'models/{model_name}/{model_name}_optimization2.pkl')  #保存模型

# 计算分折交叉验证结果
results = cross_validate(model,X,y,cv=kf, scoring=scoring, return_train_score=False)
# 输出分折交叉验证结果
for i, (r2, mae, mape) in enumerate(zip(results['test_r2'],results['test_MAE'], results['test_MAPE']), start=1):
    # 记录不同折的分数结果
    r2_scores_ls.append(r2)
    mape_scores.append(-mape)
    mae_scores.append(-mae)
    # print(f"折 {i}: r2={r2:.3f},MAE = {-mae:.3f}, MAPE = {-mape:.3f}")

current_r2_score=np.mean(r2_scores_ls)
# scores = cross_val_score(model, X, y_binned, cv=skf, scoring=make_scorer(r2_score))
# print('r2_scores:',scores)
# current_r2_score=scores.mean()
# 只对更优的结果进行保存
# 记录模型的特征重要性
feature_importance_final=model.feature_importances_
importance_dict = dict(zip(feature_importance_final, feature_names))
importance_tuplelist = [(abs(value), feature) for (value, feature) in importance_dict.items()]
importance_sorted = sorted(importance_tuplelist, reverse=True)
x_feature = [j for (i, j) in importance_sorted]
y_importance_value = [i for (i, j) in importance_sorted]
print(f'{model_name}_tree_feature_names(top15):-------+')
for i in x_feature[:15]:
    print(i)
print(f'average_{model_name}_feature_values(top15):')
for i in y_importance_value[:15]:
    print(i)
#记录更优的分数
best_r2_score=current_r2_score
best_mae_score=np.mean(mae_scores)
best_mape_score=np.mean(mape_scores)

print('平均 R2 分数:', best_r2_score)
print('平均 MAPE 分数:', best_mape_score * 100)  # 转换为百分比
print('平均MAE分数:',best_mae_score)

xgboost_tree_feature_names(top15):-------+
Interant electrons
Shear modulus local mismatch
MagpieData maximum Column
MagpieData range GSvolume_pa
Calculated Yield Strength
MagpieData mean AtomicWeight
Interant d electrons
MagpieData mean MeltingT
MagpieData avg_dev MendeleevNumber
MagpieData minimum Electronegativity
Ca fraction
MagpieData maximum NdValence
Configuration entropy
MagpieData avg_dev Electronegativity
MagpieData mean CovalentRadius
average_xgboost_feature_values(top15):
0.20819502
0.10328148
0.045753635
0.038632844
0.037108507
0.029702295
0.02840077
0.028347692
0.025465745
0.024746524
0.023306701
0.02064182
0.019914094
0.019068617
0.018637
平均 R2 分数: 0.6812609473567581
平均 MAPE 分数: 21.47500881558274
平均MAE分数: 33.35954905987891


再进一步更新参数

In [4]:
model.get_params()

{'objective': 'reg:squarederror',
 'base_score': None,
 'booster': None,
 'callbacks': None,
 'colsample_bylevel': None,
 'colsample_bynode': None,
 'colsample_bytree': None,
 'device': None,
 'early_stopping_rounds': None,
 'enable_categorical': False,
 'eval_metric': None,
 'feature_types': None,
 'gamma': None,
 'grow_policy': None,
 'importance_type': None,
 'interaction_constraints': None,
 'learning_rate': None,
 'max_bin': None,
 'max_cat_threshold': None,
 'max_cat_to_onehot': None,
 'max_delta_step': None,
 'max_depth': None,
 'max_leaves': None,
 'min_child_weight': None,
 'missing': nan,
 'monotone_constraints': None,
 'multi_strategy': None,
 'n_estimators': None,
 'n_jobs': None,
 'num_parallel_tree': None,
 'random_state': 200,
 'reg_alpha': None,
 'reg_lambda': None,
 'sampling_method': None,
 'scale_pos_weight': None,
 'subsample': None,
 'tree_method': None,
 'validate_parameters': None,
 'verbosity': None}

RF

In [28]:
from sklearn.model_selection import GridSearchCV

# 这里采用rf算法进行预测
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold,train_test_split,cross_validate,cross_val_score,KFold
from sklearn.metrics import r2_score, mean_absolute_percentage_error,mean_absolute_error,make_scorer
from xgboost import XGBRegressor
from sklearn.ensemble import RandomForestRegressor
from utils import create_empty_ls,create_dir,mycopyfile,mycopydir
import joblib
from openpyxl import load_workbook
import os
"""
路径的配置和选择
"""
model_name='rf'

save_model=True
del_0qf=True #是否删除0屈服强度数据
main_path=f'models/{model_name}' # 这里是模型所在的文件位置
create_dir(main_path,is_mainpath=True)
# excel='index/xgboost_index.xlsx'
# 如果分数表格不存在就新建一个
score_path='scores_with_best_parameters.xlsx'
if os.path.exists(score_path):
    print(f"The file '{score_path}' exists.")
else:
    print(f"The file '{score_path}' does not exist.")
    df = pd.DataFrame(columns=['Model', 'R^2', 'MAPE', 'MAE'])
    df.to_excel(score_path, index=False)


"""
文件读取部分
"""
# 读取 Excel 文件
df = pd.read_excel('train_set_new.xlsx', index_col=0)  # 引入这一列之后，原本的第一列就是一个索引，第0列会从有意义的列开始
# 删除包含空值的行
if del_0qf:
    df = df[df['屈服强度'] != 0].reset_index(drop=True)
    # df = df[df['计算晶界'] != 0].reset_index(drop=True)
    # df = df[df['计算固溶'] != 0].reset_index(drop=True)
display(df)
np.random.seed(200) #设置随机数种子
feature_names = df.drop(columns=['Precipitate Distribution',
       '屈服强度', '抗拉强度 (UTS)', '追踪编号']).columns
X = df.drop(columns=['Precipitate Distribution',
       '屈服强度', '抗拉强度 (UTS)', '追踪编号'])  # 特征: 最后四列之前的所有列
y = df.iloc[:, -3]  # 目标: 倒数第四列
# 将目标变量分箱为10个类别
bins = np.linspace(y.min(), y.max(), 11)
y_binned = np.digitize(y, bins)
# print('y_binned',y_binned)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2,random_state=200)

"""
设置初始化
"""
n_splits=5
# 初始化StratifiedKFold
kf = KFold(n_splits=5, shuffle=True, random_state=200)# 注意这里修改了random_state以确保每次初始化不同

# 初始化列表以存储每个折叠的分数
# 定义评估指标
scoring = {
    'r2':make_scorer(r2_score,greater_is_better=True),
    'MAE': make_scorer(mean_absolute_error, greater_is_better=False),
    'MAPE': make_scorer(mean_absolute_percentage_error, greater_is_better=False)
}
r2_scores_ls = []
mape_scores = []
mae_scores=[]
best_r2_score = float('-inf')
best_mae_score= float('-inf')
best_mape_score= float('-inf')
ls1=['fold'+str(i) for i in range(1,n_splits+1)]
fold_dict=dict(zip(ls1,[0 for i in range(0,len(ls1))]))

# 可修改
parameters={'max_depth':[n for n in range(3,8)], # 最大深度5,7,9
            'n_estimators':[25*n for n in range(3,8)],# DT个数 50,100,150,200
            'min_samples_leaf':[1], # 叶子结点所需最小样本数
            'min_samples_split':[2],#分裂最小样本数
            'max_features':['sqrt','log2',1],
            'criterion':['gini','squared_error']
            } #最大特征数量

# 可修改
model = RandomForestRegressor(random_state=200)  # 注意这里修改了random_state以确保每次初始化不同
grid_search=GridSearchCV(model,parameters,scoring='r2',cv=kf) # 选择r2作为性能评估参数
grid_search.fit(X_train,y_train)
best_parameter=grid_search.best_estimator_


# 可修改
model = RandomForestRegressor(random_state=best_parameter.random_state, min_samples_leaf=best_parameter.min_samples_leaf,min_samples_split=best_parameter.min_samples_split,max_depth=best_parameter.max_depth,max_features=best_parameter.max_features,criterion=best_parameter.criterion)  # 注意这里修改了random_state以确保每次初始化不同
model.fit(X_train, y_train)
print(model.get_params())
# 计算分折交叉验证结果
results = cross_validate(model,X,y,cv=kf, scoring=scoring, return_train_score=False)
# 输出分折交叉验证结果
for i, (r2, mae, mape) in enumerate(zip(results['test_r2'],results['test_MAE'], results['test_MAPE']), start=1):
    # 记录不同折的分数结果
    r2_scores_ls.append(r2)
    mape_scores.append(-mape)
    mae_scores.append(-mae)
    # print(f"折 {i}: r2={r2:.3f},MAE = {-mae:.3f}, MAPE = {-mape:.3f}")

current_r2_score=np.mean(r2_scores_ls)
# scores = cross_val_score(model, X, y_binned, cv=skf, scoring=make_scorer(r2_score))
# print('r2_scores:',scores)
# current_r2_score=scores.mean()
# 只对更优的结果进行保存
# 记录模型的特征重要性
feature_importance_final=model.feature_importances_
importance_dict = dict(zip(feature_importance_final, feature_names))
importance_tuplelist = [(abs(value), feature) for (value, feature) in importance_dict.items()]
importance_sorted = sorted(importance_tuplelist, reverse=True)
x_feature = [j for (i, j) in importance_sorted]
y_importance_value = [i for (i, j) in importance_sorted]
print(f'{model_name}_tree_feature_names(top15):')
for i in x_feature[:15]:
    print(i)
print(f'average_{model_name}_feature_values(top15):')
for i in y_importance_value[:15]:
    print(i)

#记录更优的分数
best_r2_score=current_r2_score
best_mae_score=np.mean(mae_scores)
best_mape_score=np.mean(mape_scores)
if save_model:
    joblib.dump(model,   f'models/{model_name}/{model_name}_optimization1.pkl')  #保存模型
    # print(f'保存更优{model_name}模型文件') 
print('平均 R2 分数:', best_r2_score)
print('平均 MAPE 分数:', best_mape_score * 100)  # 转换为百分比
print('平均MAE分数:',best_mae_score)


    
"""
打印并输出分数至表格
"""

df = pd.read_excel(score_path)
# 添加新数据的示例
new_data = {'Model': f'{model_name}', 'R^2': best_r2_score, 'MAPE': best_mape_score * 100, 'MAE': best_mae_score}
new_data_df = pd.DataFrame([new_data])

# 使用 pd.concat() 合并原 DataFrame 和新的 DataFrame
df = pd.concat([df, new_data_df], ignore_index=True)
# 保存数据框到Excel表格
df.to_excel(score_path, index = False)

    

文件夹models/rf已存在
The file 'scores_with_best_parameters.xlsx' exists.


,MagpieData minimum Number,MagpieData maximum Number,MagpieData range Number,MagpieData mean Number,MagpieData avg_dev Number,MagpieData mode Number,MagpieData minimum MendeleevNumber,MagpieData maximum MendeleevNumber,MagpieData range MendeleevNumber,MagpieData mean MendeleevNumber,...,frac f valence electrons,Grain Size,Precipitate Distribution,Habit Plane,Calculated Solid Solution,Calculated Grain Boundary,Calculated Yield Strength,屈服强度,抗拉强度 (UTS),追踪编号
0,12,30,18,12.131276,0.247009,12,52,73,21,68.266826,...,0.000000,4.00,0,0,17.669989,194.321983,226.991971,364.0,393.0,No630-6Al-1Zn-0.15Mn -3
1,12,64,52,12.922815,1.811506,12,23,68,45,67.259700,...,0.054984,36.00,3,1,57.602249,48.504397,121.106646,125.0,200.0,No232-Mg-9Gd-1Sm-0.5Zr-2
2,12,64,52,13.780587,3.401587,12,12,69,57,66.376359,...,0.078055,50.00,3,5,94.613373,69.789053,179.402426,200.0,340.0,No16-Mg–14Gd–3Y–1.8Zn–0.5Zr -9
3,12,64,52,13.179217,2.292508,12,12,68,56,66.814372,...,0.052520,1.50,2,1,76.138523,262.520047,353.658570,290.0,280.0,No96-Mg-9.3Gd-2.7Y-0.65Ag-0.52Zr-0.18Ce -4
4,12,30,18,12.112392,0.217496,12,52,73,21,68.118900,...,0.000000,15.00,0,0,11.604694,86.484728,113.089422,148.0,280.0,No460-3Al-1Zn-0.3Mn -22
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
734,12,38,26,12.053936,0.104733,12,8,73,65,68.099036,...,0.000000,130.00,1,1,12.020723,23.509472,50.530195,58.5,220.0,No744-Mg-3Al-0.4Mn-0.05Sr
735,12,64,52,12.937303,1.825661,12,12,69,57,67.051202,...,0.033095,41.77,3,5,61.765688,53.715298,130.480986,223.0,295.0,No226-Mg-6Gd-3Y-1Zn-0.4Zr-0.7Ag-3
736,12,30,18,12.152650,0.281635,12,7,73,66,67.973850,...,0.000000,30.00,3,2,27.156102,71.029741,113.185842,91.1,144.3,No312-Mg-7.6Al-0.5Zn-1Ca-2
737,12,64,52,12.126080,0.237903,12,12,73,61,68.188783,...,0.003189,52.10,1,1,23.809563,45.036179,83.845741,125.0,217.0,No383-Mg-6Al-Zn-0.3Y-0.6Gd


{'bootstrap': True, 'ccp_alpha': 0.0, 'criterion': 'squared_error', 'max_depth': 7, 'max_features': 'sqrt', 'max_leaf_nodes': None, 'max_samples': None, 'min_impurity_decrease': 0.0, 'min_samples_leaf': 1, 'min_samples_split': 2, 'min_weight_fraction_leaf': 0.0, 'monotonic_cst': None, 'n_estimators': 100, 'n_jobs': None, 'oob_score': False, 'random_state': 200, 'verbose': 0, 'warm_start': False}
rf_tree_feature_names(top15):
Calculated Yield Strength
Calculated Grain Boundary
Grain Size
Habit Plane
frac f valence electrons
Gd fraction
Atomic weight mean
MagpieData avg_dev NfValence
MagpieData avg_dev NfUnfilled
Interant electrons
MagpieData mean NfUnfilled
MagpieData mean GSvolume_pa
Radii gamma
MagpieData mean MendeleevNumber
MagpieData mean MeltingT
average_rf_feature_values(top15):
0.08548370645905354
0.08010779263523725
0.07791413138455372
0.032879253337416
0.027543388635443818
0.027233918602586837
0.02285374741290424
0.020739102448246862
0.020357508136141744
0.020136626811053444
0

In [29]:
model = RandomForestRegressor(random_state=200)  # 注意这里修改了random_state以确保每次初始化不同
model.fit(X_train, y_train)
# 计算分折交叉验证结果
results = cross_validate(model,X,y,cv=kf, scoring=scoring, return_train_score=False)
# 输出分折交叉验证结果
for i, (r2, mae, mape) in enumerate(zip(results['test_r2'],results['test_MAE'], results['test_MAPE']), start=1):
    # 记录不同折的分数结果
    r2_scores_ls.append(r2)
    mape_scores.append(-mape)
    mae_scores.append(-mae)
    # print(f"折 {i}: r2={r2:.3f},MAE = {-mae:.3f}, MAPE = {-mape:.3f}")

current_r2_score=np.mean(r2_scores_ls)
# scores = cross_val_score(model, X, y_binned, cv=skf, scoring=make_scorer(r2_score))
# print('r2_scores:',scores)
# current_r2_score=scores.mean()
# 只对更优的结果进行保存
# 记录模型的特征重要性
feature_importance_final=model.feature_importances_
importance_dict = dict(zip(feature_importance_final, feature_names))
importance_tuplelist = [(abs(value), feature) for (value, feature) in importance_dict.items()]
importance_sorted = sorted(importance_tuplelist, reverse=True)
x_feature = [j for (i, j) in importance_sorted]
y_importance_value = [i for (i, j) in importance_sorted]
print(f'{model_name}_tree_feature_names(top15):-------+')
for i in x_feature[:15]:
    print(i)
print(f'average_{model_name}_feature_values(top15):')
for i in y_importance_value[:15]:
    print(i)
#记录更优的分数
best_r2_score=current_r2_score
best_mae_score=np.mean(mae_scores)
best_mape_score=np.mean(mape_scores)

print('平均 R2 分数:', best_r2_score)
print('平均 MAPE 分数:', best_mape_score * 100)  # 转换为百分比
print('平均MAE分数:',best_mae_score)
model.get_params()

rf_tree_feature_names(top15):-------+
Calculated Yield Strength
Grain Size
Habit Plane
Calculated Grain Boundary
Interant electrons
MagpieData avg_dev MeltingT
Lambda entropy
Radii gamma
frac f valence electrons
Interant d electrons
Zr fraction
Yang omega
mean AtomicRadius
Mg fraction
Mn fraction
average_rf_feature_values(top15):
0.37279360782287885
0.11498520137520721
0.05459789944572503
0.03546350905848006
0.02364842340761164
0.021152779330123248
0.01854824905114933
0.015401955458189487
0.011677698392683424
0.011384362113809889
0.010931370603955576
0.010570390669629762
0.009395848367317291
0.009157660254578438
0.009091918943336061
平均 R2 分数: 0.6634879878453341
平均 MAPE 分数: 24.083428987756218
平均MAE分数: 35.93314253964674


{'bootstrap': True,
 'ccp_alpha': 0.0,
 'criterion': 'squared_error',
 'max_depth': None,
 'max_features': 1.0,
 'max_leaf_nodes': None,
 'max_samples': None,
 'min_impurity_decrease': 0.0,
 'min_samples_leaf': 1,
 'min_samples_split': 2,
 'min_weight_fraction_leaf': 0.0,
 'monotonic_cst': None,
 'n_estimators': 100,
 'n_jobs': None,
 'oob_score': False,
 'random_state': 200,
 'verbose': 0,
 'warm_start': False}

In [30]:
joblib.dump(model,   f'models/{model_name}/{model_name}_optimization2.pkl')  #保存模型

['models/rf/rf_optimization2.pkl']

再次得到最优参数，发现效果并不好

In [31]:
from sklearn.model_selection import GridSearchCV

# 这里采用rf算法进行预测
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold,train_test_split,cross_validate,cross_val_score,KFold
from sklearn.metrics import r2_score, mean_absolute_percentage_error,mean_absolute_error,make_scorer
from xgboost import XGBRegressor
from sklearn.ensemble import RandomForestRegressor
from utils import create_empty_ls,create_dir,mycopyfile,mycopydir
import joblib
from openpyxl import load_workbook
import os
"""
路径的配置和选择
"""
model_name='rf'

save_model=True
del_0qf=True #是否删除0屈服强度数据
main_path=f'models/{model_name}' # 这里是模型所在的文件位置
create_dir(main_path,is_mainpath=True)
# excel='index/xgboost_index.xlsx'
# 如果分数表格不存在就新建一个
score_path='scores_with_best_parameters.xlsx'
if os.path.exists(score_path):
    print(f"The file '{score_path}' exists.")
else:
    print(f"The file '{score_path}' does not exist.")
    df = pd.DataFrame(columns=['Model', 'R^2', 'MAPE', 'MAE'])
    df.to_excel(score_path, index=False)


"""
文件读取部分
"""
# 读取 Excel 文件
df = pd.read_excel('train_set_new.xlsx', index_col=0)  # 引入这一列之后，原本的第一列就是一个索引，第0列会从有意义的列开始
# 删除包含空值的行
if del_0qf:
    df = df[df['屈服强度'] != 0].reset_index(drop=True)
    # df = df[df['计算晶界'] != 0].reset_index(drop=True)
    # df = df[df['计算固溶'] != 0].reset_index(drop=True)
display(df)
np.random.seed(200) #设置随机数种子
feature_names = df.columns[:-4]  # 特征名为屈服强度（倒数第四列）之前的部分
X = df.iloc[:, :-4]  # 特征: 最后四列之前的所有列
y = df.iloc[:, -4]  # 目标: 倒数第四列
# 将目标变量分箱为10个类别
bins = np.linspace(y.min(), y.max(), 11)
y_binned = np.digitize(y, bins)
# print('y_binned',y_binned)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2,random_state=200)

"""
设置初始化
"""
n_splits=5
# 初始化StratifiedKFold
kf = KFold(n_splits=5, shuffle=True, random_state=200)# 注意这里修改了random_state以确保每次初始化不同

# 初始化列表以存储每个折叠的分数
# 定义评估指标
scoring = {
    'r2':make_scorer(r2_score,greater_is_better=True),
    'MAE': make_scorer(mean_absolute_error, greater_is_better=False),
    'MAPE': make_scorer(mean_absolute_percentage_error, greater_is_better=False)
}
r2_scores_ls = []
mape_scores = []
mae_scores=[]
best_r2_score = float('-inf')
best_mae_score= float('-inf')
best_mape_score= float('-inf')
ls1=['fold'+str(i) for i in range(1,n_splits+1)]
fold_dict=dict(zip(ls1,[0 for i in range(0,len(ls1))]))

# 可修改
parameters={'max_depth':[n for n in range(3,8,2)], # 最大深度5,7,9
            'n_estimators':[100,125,150],# DT个数 50,100,150,200
            'min_samples_leaf':[1], # 叶子结点所需最小样本数
            'min_samples_split':[2],#分裂最小样本数
            'max_features':[1],
            'criterion':['squared_error']
            } #最大特征数量

# 可修改
model = RandomForestRegressor(random_state=200)  # 注意这里修改了random_state以确保每次初始化不同
grid_search=GridSearchCV(model,parameters,scoring='r2',cv=kf) # 选择r2作为性能评估参数
grid_search.fit(X_train,y_train)
best_parameter=grid_search.best_estimator_


# 可修改
model = RandomForestRegressor(random_state=best_parameter.random_state, min_samples_leaf=best_parameter.min_samples_leaf,min_samples_split=best_parameter.min_samples_split,max_depth=best_parameter.max_depth,max_features=best_parameter.max_features,criterion=best_parameter.criterion)  # 注意这里修改了random_state以确保每次初始化不同
model.fit(X_train, y_train)
print(model.get_params())
# 计算分折交叉验证结果
results = cross_validate(model,X,y,cv=kf, scoring=scoring, return_train_score=False)
# 输出分折交叉验证结果
for i, (r2, mae, mape) in enumerate(zip(results['test_r2'],results['test_MAE'], results['test_MAPE']), start=1):
    # 记录不同折的分数结果
    r2_scores_ls.append(r2)
    mape_scores.append(-mape)
    mae_scores.append(-mae)
    # print(f"折 {i}: r2={r2:.3f},MAE = {-mae:.3f}, MAPE = {-mape:.3f}")

current_r2_score=np.mean(r2_scores_ls)
# scores = cross_val_score(model, X, y_binned, cv=skf, scoring=make_scorer(r2_score))
# print('r2_scores:',scores)
# current_r2_score=scores.mean()
# 只对更优的结果进行保存
# 记录模型的特征重要性
feature_importance_final=model.feature_importances_
importance_dict = dict(zip(feature_importance_final, feature_names))
importance_tuplelist = [(abs(value), feature) for (value, feature) in importance_dict.items()]
importance_sorted = sorted(importance_tuplelist, reverse=True)
x_feature = [j for (i, j) in importance_sorted]
y_importance_value = [i for (i, j) in importance_sorted]
print(f'{model_name}_tree_feature_names(top15):')
for i in x_feature[:15]:
    print(i)
print(f'average_{model_name}_feature_values(top15):')
for i in y_importance_value[:15]:
    print(i)

#记录更优的分数
best_r2_score=current_r2_score
best_mae_score=np.mean(mae_scores)
best_mape_score=np.mean(mape_scores)
if save_model:
    joblib.dump(model,   f'models/{model_name}/{model_name}_optimization3.pkl')  #保存模型
    # print(f'保存更优{model_name}模型文件') 
print('平均 R2 分数:', best_r2_score)
print('平均 MAPE 分数:', best_mape_score * 100)  # 转换为百分比
print('平均MAE分数:',best_mae_score)



"""
打印并输出分数至表格
"""

df = pd.read_excel(score_path)
# 添加新数据的示例
new_data = {'Model': f'{model_name}', 'R^2': best_r2_score, 'MAPE': best_mape_score * 100, 'MAE': best_mae_score}
new_data_df = pd.DataFrame([new_data])

# 使用 pd.concat() 合并原 DataFrame 和新的 DataFrame
df = pd.concat([df, new_data_df], ignore_index=True)
# 保存数据框到Excel表格
df.to_excel(score_path, index = False)



文件夹models/rf已存在
The file 'scores_with_best_parameters.xlsx' exists.


,MagpieData minimum Number,MagpieData maximum Number,MagpieData range Number,MagpieData mean Number,MagpieData avg_dev Number,MagpieData mode Number,MagpieData minimum MendeleevNumber,MagpieData maximum MendeleevNumber,MagpieData range MendeleevNumber,MagpieData mean MendeleevNumber,...,frac f valence electrons,Grain Size,Precipitate Distribution,Habit Plane,Calculated Solid Solution,Calculated Grain Boundary,Calculated Yield Strength,屈服强度,抗拉强度 (UTS),追踪编号
0,12,30,18,12.131276,0.247009,12,52,73,21,68.266826,...,0.000000,4.00,0,0,17.669989,194.321983,226.991971,364.0,393.0,No630-6Al-1Zn-0.15Mn -3
1,12,64,52,12.922815,1.811506,12,23,68,45,67.259700,...,0.054984,36.00,3,1,57.602249,48.504397,121.106646,125.0,200.0,No232-Mg-9Gd-1Sm-0.5Zr-2
2,12,64,52,13.780587,3.401587,12,12,69,57,66.376359,...,0.078055,50.00,3,5,94.613373,69.789053,179.402426,200.0,340.0,No16-Mg–14Gd–3Y–1.8Zn–0.5Zr -9
3,12,64,52,13.179217,2.292508,12,12,68,56,66.814372,...,0.052520,1.50,2,1,76.138523,262.520047,353.658570,290.0,280.0,No96-Mg-9.3Gd-2.7Y-0.65Ag-0.52Zr-0.18Ce -4
4,12,30,18,12.112392,0.217496,12,52,73,21,68.118900,...,0.000000,15.00,0,0,11.604694,86.484728,113.089422,148.0,280.0,No460-3Al-1Zn-0.3Mn -22
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
734,12,38,26,12.053936,0.104733,12,8,73,65,68.099036,...,0.000000,130.00,1,1,12.020723,23.509472,50.530195,58.5,220.0,No744-Mg-3Al-0.4Mn-0.05Sr
735,12,64,52,12.937303,1.825661,12,12,69,57,67.051202,...,0.033095,41.77,3,5,61.765688,53.715298,130.480986,223.0,295.0,No226-Mg-6Gd-3Y-1Zn-0.4Zr-0.7Ag-3
736,12,30,18,12.152650,0.281635,12,7,73,66,67.973850,...,0.000000,30.00,3,2,27.156102,71.029741,113.185842,91.1,144.3,No312-Mg-7.6Al-0.5Zn-1Ca-2
737,12,64,52,12.126080,0.237903,12,12,73,61,68.188783,...,0.003189,52.10,1,1,23.809563,45.036179,83.845741,125.0,217.0,No383-Mg-6Al-Zn-0.3Y-0.6Gd


{'bootstrap': True, 'ccp_alpha': 0.0, 'criterion': 'squared_error', 'max_depth': 7, 'max_features': 1, 'max_leaf_nodes': None, 'max_samples': None, 'min_impurity_decrease': 0.0, 'min_samples_leaf': 1, 'min_samples_split': 2, 'min_weight_fraction_leaf': 0.0, 'monotonic_cst': None, 'n_estimators': 100, 'n_jobs': None, 'oob_score': False, 'random_state': 200, 'verbose': 0, 'warm_start': False}
rf_tree_feature_names(top15):
Grain Size
Calculated Grain Boundary
MagpieData mean NdValence
MagpieData mean MendeleevNumber
mean Number
MagpieData avg_dev Row
MagpieData mean Number
Shear modulus strength model
frac f valence electrons
Atomic weight mean
MagpieData avg_dev NValence
MagpieData avg_dev AtomicWeight
mean AtomicRadius
MagpieData mean NUnfilled
Mn fraction
average_rf_feature_values(top15):
0.07536769741295536
0.04047405686683142
0.02327585199638279
0.015402027481419597
0.015168555013338726
0.014829930636409763
0.01481555041066438
0.014463642718022312
0.0140161018774845
0.013641700032297

GBoost

In [9]:
from sklearn.model_selection import GridSearchCV

# 这里采用xgboost算法进行预测
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold,train_test_split,cross_validate,cross_val_score,KFold
from sklearn.metrics import r2_score, mean_absolute_percentage_error,mean_absolute_error,make_scorer
from xgboost import XGBRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import GradientBoostingRegressor  # 导入GradientBoostingRegressor
from sklearn.inspection import permutation_importance
from utils import create_empty_ls,create_dir,mycopyfile,mycopydir
import joblib
from openpyxl import load_workbook
import os
"""
路径的配置和选择
"""
model_name='gboost'

save_model=True
del_0qf=True #是否删除0屈服强度数据
main_path=f'models/{model_name}' # 这里是模型所在的文件位置
create_dir(main_path,is_mainpath=True)
# excel='index/xgboost_index.xlsx'
# 如果分数表格不存在就新建一个
score_path='scores_with_best_parameters.xlsx'
if os.path.exists(score_path):
    print(f"The file '{score_path}' exists.")
else:
    print(f"The file '{score_path}' does not exist.")
    df = pd.DataFrame(columns=['Model', 'R^2', 'MAPE', 'MAE'])
    df.to_excel(score_path, index=False)

"""
文件读取部分
"""
# 读取 Excel 文件
df = pd.read_excel('train_set_new.xlsx', index_col=0)  # 引入这一列之后，原本的第一列就是一个索引，第0列会从有意义的列开始
# 删除包含空值的行
if del_0qf:
    df = df[df['屈服强度'] != 0].reset_index(drop=True)
    # df = df[df['计算晶界'] != 0].reset_index(drop=True)
    # df = df[df['计算固溶'] != 0].reset_index(drop=True)
display(df)
np.random.seed(200) #设置随机数种子
feature_names = df.drop(columns=['Precipitate Distribution',
       '屈服强度', '抗拉强度 (UTS)', '追踪编号']).columns
X = df.drop(columns=['Precipitate Distribution',
       '屈服强度', '抗拉强度 (UTS)', '追踪编号'])  # 特征: 最后四列之前的所有列
y = df.iloc[:, -3]  # 目标: 倒数第四列
# 将目标变量分箱为10个类别
bins = np.linspace(y.min(), y.max(), 11)
y_binned = np.digitize(y, bins)
# print('y_binned',y_binned)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2,random_state=200)

"""
设置初始化
"""
n_splits=5
# 初始化StratifiedKFold
kf = KFold(n_splits=5, shuffle=True, random_state=200)# 注意这里修改了random_state以确保每次初始化不同

# 初始化列表以存储每个折叠的分数
# 定义评估指标
scoring = {
    'r2':make_scorer(r2_score,greater_is_better=True),
    'MAE': make_scorer(mean_absolute_error, greater_is_better=False),
    'MAPE': make_scorer(mean_absolute_percentage_error, greater_is_better=False)
}
r2_scores_ls = []
mape_scores = []
mae_scores=[]
best_r2_score = float('-inf')
best_mae_score= float('-inf')
best_mape_score= float('-inf')
ls1=['fold'+str(i) for i in range(1,n_splits+1)]
fold_dict=dict(zip(ls1,[0 for i in range(0,len(ls1))]))

# 可修改
parameters={'max_depth':[n for n in range(3,8,2)], # 最大深度5,7,9
            'n_estimators':[50*n for n in range(1,5)],# DT个数 50,100,150,200
            'min_samples_leaf':[1,2], # 叶子结点所需最小样本数
            'min_samples_split':[2,3,4],#分裂最小样本数
            'max_features':['auto','sqrt','log2','None'],
            'learning_rate':[0.01,0.1,0.2], 
            } #最大特征数量

# 可修改
model = GradientBoostingRegressor(random_state=200) # 注意这里修改了random_state以确保每次初始化不同
grid_search=GridSearchCV(model,parameters,scoring='r2',cv=kf) # 选择r2作为性能评估参数
grid_search.fit(X_train,y_train)
best_parameter=grid_search.best_estimator_


# 可修改
model = GradientBoostingRegressor(n_estimators=best_parameter.n_estimators,min_samples_leaf=best_parameter.min_samples_leaf,min_samples_split=best_parameter.min_samples_split,max_depth=best_parameter.max_depth,max_features=best_parameter.max_features,learning_rate=best_parameter.learning_rate)  # 注意这里修改了random_state以确保每次初始化不同
model.fit(X_train, y_train)
print(model.get_params())
# 计算分折交叉验证结果
results = cross_validate(model,X,y,cv=kf, scoring=scoring, return_train_score=False)
# 输出分折交叉验证结果
for i, (r2, mae, mape) in enumerate(zip(results['test_r2'],results['test_MAE'], results['test_MAPE']), start=1):
    # 记录不同折的分数结果
    r2_scores_ls.append(r2)
    mape_scores.append(-mape)
    mae_scores.append(-mae)
    # print(f"折 {i}: r2={r2:.3f},MAE = {-mae:.3f}, MAPE = {-mape:.3f}")

current_r2_score=np.mean(r2_scores_ls)
# scores = cross_val_score(model, X, y_binned, cv=skf, scoring=make_scorer(r2_score))
# print('r2_scores:',scores)
# current_r2_score=scores.mean()
# 只对更优的结果进行保存
# 记录模型的特征重要性
feature_importance_final=model.feature_importances_
importance_dict = dict(zip(feature_importance_final, feature_names))
importance_tuplelist = [(abs(value), feature) for (value, feature) in importance_dict.items()]
importance_sorted = sorted(importance_tuplelist, reverse=True)
x_feature = [j for (i, j) in importance_sorted]
y_importance_value = [i for (i, j) in importance_sorted]
print(f'{model_name}_tree_feature_names(top15):')
for i in x_feature[:15]:
    print(i)
print(f'average_{model_name}_feature_values(top15):')
for i in y_importance_value[:15]:
    print(i)

#记录更优的分数
best_r2_score=current_r2_score
best_mae_score=np.mean(mae_scores)
best_mape_score=np.mean(mape_scores)
if save_model:
    joblib.dump(model,   f'models/{model_name}/{model_name}_optimization1.pkl')  #保存模型
    # print(f'保存更优{model_name}模型文件') 
print('平均 R2 分数:', best_r2_score)
print('平均 MAPE 分数:', best_mape_score * 100)  # 转换为百分比
print('平均MAE分数:',best_mae_score)
    
"""
打印并输出分数至表格
"""

df = pd.read_excel(score_path)
# 添加新数据的示例
new_data = {'Model': f'{model_name}', 'R^2': best_r2_score, 'MAPE': best_mape_score * 100, 'MAE': best_mae_score}
new_data_df = pd.DataFrame([new_data])

# 使用 pd.concat() 合并原 DataFrame 和新的 DataFrame
df = pd.concat([df, new_data_df], ignore_index=True)
# 保存数据框到Excel表格
df.to_excel(score_path, index = False)


    

文件夹models/gboost已存在
The file 'scores_with_best_parameters.xlsx' exists.


,MagpieData minimum Number,MagpieData maximum Number,MagpieData range Number,MagpieData mean Number,MagpieData avg_dev Number,MagpieData mode Number,MagpieData minimum MendeleevNumber,MagpieData maximum MendeleevNumber,MagpieData range MendeleevNumber,MagpieData mean MendeleevNumber,...,frac f valence electrons,Grain Size,Precipitate Distribution,Habit Plane,Calculated Solid Solution,Calculated Grain Boundary,Calculated Yield Strength,屈服强度,抗拉强度 (UTS),追踪编号
0,12,30,18,12.131276,0.247009,12,52,73,21,68.266826,...,0.000000,4.00,0,0,17.669989,194.321983,226.991971,364.0,393.0,No630-6Al-1Zn-0.15Mn -3
1,12,64,52,12.922815,1.811506,12,23,68,45,67.259700,...,0.054984,36.00,3,1,57.602249,48.504397,121.106646,125.0,200.0,No232-Mg-9Gd-1Sm-0.5Zr-2
2,12,64,52,13.780587,3.401587,12,12,69,57,66.376359,...,0.078055,50.00,3,5,94.613373,69.789053,179.402426,200.0,340.0,No16-Mg–14Gd–3Y–1.8Zn–0.5Zr -9
3,12,64,52,13.179217,2.292508,12,12,68,56,66.814372,...,0.052520,1.50,2,1,76.138523,262.520047,353.658570,290.0,280.0,No96-Mg-9.3Gd-2.7Y-0.65Ag-0.52Zr-0.18Ce -4
4,12,30,18,12.112392,0.217496,12,52,73,21,68.118900,...,0.000000,15.00,0,0,11.604694,86.484728,113.089422,148.0,280.0,No460-3Al-1Zn-0.3Mn -22
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
734,12,38,26,12.053936,0.104733,12,8,73,65,68.099036,...,0.000000,130.00,1,1,12.020723,23.509472,50.530195,58.5,220.0,No744-Mg-3Al-0.4Mn-0.05Sr
735,12,64,52,12.937303,1.825661,12,12,69,57,67.051202,...,0.033095,41.77,3,5,61.765688,53.715298,130.480986,223.0,295.0,No226-Mg-6Gd-3Y-1Zn-0.4Zr-0.7Ag-3
736,12,30,18,12.152650,0.281635,12,7,73,66,67.973850,...,0.000000,30.00,3,2,27.156102,71.029741,113.185842,91.1,144.3,No312-Mg-7.6Al-0.5Zn-1Ca-2
737,12,64,52,12.126080,0.237903,12,12,73,61,68.188783,...,0.003189,52.10,1,1,23.809563,45.036179,83.845741,125.0,217.0,No383-Mg-6Al-Zn-0.3Y-0.6Gd


{'alpha': 0.9, 'ccp_alpha': 0.0, 'criterion': 'friedman_mse', 'init': None, 'learning_rate': 0.1, 'loss': 'squared_error', 'max_depth': 3, 'max_features': 'sqrt', 'max_leaf_nodes': None, 'min_impurity_decrease': 0.0, 'min_samples_leaf': 1, 'min_samples_split': 4, 'min_weight_fraction_leaf': 0.0, 'n_estimators': 150, 'n_iter_no_change': None, 'random_state': None, 'subsample': 1.0, 'tol': 0.0001, 'validation_fraction': 0.1, 'verbose': 0, 'warm_start': False}
gboost_tree_feature_names(top15):
Calculated Grain Boundary
Calculated Yield Strength
Grain Size
Habit Plane
mean AtomicRadius
MagpieData mean MendeleevNumber
MagpieData avg_dev NfUnfilled
MagpieData mean NfValence
MagpieData maximum GSmagmom
frac f valence electrons
MagpieData avg_dev Number
MagpieData avg_dev MendeleevNumber
Shear modulus strength model
MagpieData avg_dev GSmagmom
MagpieData range NUnfilled
average_gboost_feature_values(top15):
0.11283118229713211
0.09497099711377265
0.0887861635481043
0.04928522243700119
0.045637

In [10]:
model = GradientBoostingRegressor(random_state=200)  # 注意这里修改了random_state以确保每次初始化不同
model.fit(X_train, y_train)
# 计算分折交叉验证结果
results = cross_validate(model,X,y,cv=kf, scoring=scoring, return_train_score=False)
# 输出分折交叉验证结果
for i, (r2, mae, mape) in enumerate(zip(results['test_r2'],results['test_MAE'], results['test_MAPE']), start=1):
    # 记录不同折的分数结果
    r2_scores_ls.append(r2)
    mape_scores.append(-mape)
    mae_scores.append(-mae)
    # print(f"折 {i}: r2={r2:.3f},MAE = {-mae:.3f}, MAPE = {-mape:.3f}")

current_r2_score=np.mean(r2_scores_ls)
# scores = cross_val_score(model, X, y_binned, cv=skf, scoring=make_scorer(r2_score))
# print('r2_scores:',scores)
# current_r2_score=scores.mean()
# 只对更优的结果进行保存
# 记录模型的特征重要性
feature_importance_final=model.feature_importances_
importance_dict = dict(zip(feature_importance_final, feature_names))
importance_tuplelist = [(abs(value), feature) for (value, feature) in importance_dict.items()]
importance_sorted = sorted(importance_tuplelist, reverse=True)
x_feature = [j for (i, j) in importance_sorted]
y_importance_value = [i for (i, j) in importance_sorted]
print(f'{model_name}_tree_feature_names(top15):-------+')
for i in x_feature[:15]:
    print(i)
print(f'average_{model_name}_feature_values(top15):')
for i in y_importance_value[:15]:
    print(i)
#记录更优的分数
best_r2_score=current_r2_score
best_mae_score=np.mean(mae_scores)
best_mape_score=np.mean(mape_scores)

print('平均 R2 分数:', best_r2_score)
print('平均 MAPE 分数:', best_mape_score * 100)  # 转换为百分比
print('平均MAE分数:',best_mae_score)
model.get_params()

gboost_tree_feature_names(top15):-------+
Calculated Yield Strength
Grain Size
Habit Plane
Interant electrons
MagpieData avg_dev MeltingT
Mn fraction
Zr fraction
frac f valence electrons
Radii gamma
MagpieData avg_dev Column
MagpieData avg_dev NdUnfilled
MagpieData avg_dev MendeleevNumber
Calculated Solid Solution
MagpieData mean NdUnfilled
mean Row
average_gboost_feature_values(top15):
0.2944618072441523
0.18227691909513266
0.08149985383124486
0.036853945066644654
0.034283853809801
0.03134643088106182
0.01944721033474525
0.019376281107912598
0.01919143218019831
0.018658146938146655
0.016652417216762305
0.01594320897623932
0.013976938348114767
0.013650058816977808
0.011305398880009638
平均 R2 分数: 0.703978810229329
平均 MAPE 分数: 22.493587339921532
平均MAE分数: 33.611127062876946


{'alpha': 0.9,
 'ccp_alpha': 0.0,
 'criterion': 'friedman_mse',
 'init': None,
 'learning_rate': 0.1,
 'loss': 'squared_error',
 'max_depth': 3,
 'max_features': None,
 'max_leaf_nodes': None,
 'min_impurity_decrease': 0.0,
 'min_samples_leaf': 1,
 'min_samples_split': 2,
 'min_weight_fraction_leaf': 0.0,
 'n_estimators': 100,
 'n_iter_no_change': None,
 'random_state': 200,
 'subsample': 1.0,
 'tol': 0.0001,
 'validation_fraction': 0.1,
 'verbose': 0,
 'warm_start': False}

In [11]:
joblib.dump(model,'models/gboost/gboost_best.pkl')

['models/gboost/gboost_best.pkl']

Gboost参数再优化

In [12]:
from sklearn.model_selection import GridSearchCV

# 这里采用xgboost算法进行预测
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold,train_test_split,cross_validate,cross_val_score,KFold
from sklearn.metrics import r2_score, mean_absolute_percentage_error,mean_absolute_error,make_scorer
from xgboost import XGBRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import GradientBoostingRegressor  # 导入GradientBoostingRegressor
from sklearn.inspection import permutation_importance
from utils import create_empty_ls,create_dir,mycopyfile,mycopydir
import joblib
from openpyxl import load_workbook
import os
"""
路径的配置和选择
"""
model_name='gboost'

save_model=True
del_0qf=True #是否删除0屈服强度数据
main_path=f'models/{model_name}' # 这里是模型所在的文件位置
create_dir(main_path,is_mainpath=True)
# excel='index/xgboost_index.xlsx'
# 如果分数表格不存在就新建一个
score_path='scores_with_best_parameters.xlsx'
if os.path.exists(score_path):
    print(f"The file '{score_path}' exists.")
else:
    print(f"The file '{score_path}' does not exist.")
    df = pd.DataFrame(columns=['Model', 'R^2', 'MAPE', 'MAE'])
    df.to_excel(score_path, index=False)


"""
文件读取部分
"""
# 读取 Excel 文件
df = pd.read_excel('train_set_new.xlsx', index_col=0)  # 引入这一列之后，原本的第一列就是一个索引，第0列会从有意义的列开始
# 删除包含空值的行
if del_0qf:
    df = df[df['屈服强度'] != 0].reset_index(drop=True)
    # df = df[df['计算晶界'] != 0].reset_index(drop=True)
    # df = df[df['计算固溶'] != 0].reset_index(drop=True)
display(df)
np.random.seed(200) #设置随机数种子
feature_names = df.drop(columns=['Precipitate Distribution',
       '屈服强度', '抗拉强度 (UTS)', '追踪编号']).columns
X = df.drop(columns=['Precipitate Distribution',
       '屈服强度', '抗拉强度 (UTS)', '追踪编号'])  # 特征: 最后四列之前的所有列
y = df.iloc[:, -3]  # 目标: 倒数第四列
# 将目标变量分箱为10个类别
bins = np.linspace(y.min(), y.max(), 11)
y_binned = np.digitize(y, bins)
# print('y_binned',y_binned)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2,random_state=200)

"""
设置初始化
"""
n_splits=5
# 初始化StratifiedKFold
kf = KFold(n_splits=5, shuffle=True, random_state=200)# 注意这里修改了random_state以确保每次初始化不同

# 初始化列表以存储每个折叠的分数
# 定义评估指标
scoring = {
    'r2':make_scorer(r2_score,greater_is_better=True),
    'MAE': make_scorer(mean_absolute_error, greater_is_better=False),
    'MAPE': make_scorer(mean_absolute_percentage_error, greater_is_better=False)
}
r2_scores_ls = []
mape_scores = []
mae_scores=[]
best_r2_score = float('-inf')
best_mae_score= float('-inf')
best_mape_score= float('-inf')
ls1=['fold'+str(i) for i in range(1,n_splits+1)]
fold_dict=dict(zip(ls1,[0 for i in range(0,len(ls1))]))

# 可修改
parameters={'max_depth':[n for n in range(3,6,1)], # 最大深度5,7,9
            'n_estimators':[25*n for n in range(3,5)],# DT个数 50,100,150,200
            'min_samples_leaf':[1], # 叶子结点所需最小样本数
            'min_samples_split':[2],#分裂最小样本数
            'max_features':['auto','sqrt','log2','None'],
            'learning_rate':np.arange(0.08,0.15,0.01), 
            } #最大特征数量

# 可修改
model = GradientBoostingRegressor(random_state=200) # 注意这里修改了random_state以确保每次初始化不同
grid_search=GridSearchCV(model,parameters,scoring='r2',cv=kf) # 选择r2作为性能评估参数
grid_search.fit(X_train,y_train)
best_parameter=grid_search.best_estimator_


# 可修改
model = GradientBoostingRegressor(n_estimators=best_parameter.n_estimators,min_samples_leaf=best_parameter.min_samples_leaf,min_samples_split=best_parameter.min_samples_split,max_depth=best_parameter.max_depth,max_features=best_parameter.max_features,learning_rate=best_parameter.learning_rate)  # 注意这里修改了random_state以确保每次初始化不同
model.fit(X_train, y_train)
print(model.get_params())
# 计算分折交叉验证结果
results = cross_validate(model,X,y,cv=kf, scoring=scoring, return_train_score=False)
# 输出分折交叉验证结果
for i, (r2, mae, mape) in enumerate(zip(results['test_r2'],results['test_MAE'], results['test_MAPE']), start=1):
    # 记录不同折的分数结果
    r2_scores_ls.append(r2)
    mape_scores.append(-mape)
    mae_scores.append(-mae)
    # print(f"折 {i}: r2={r2:.3f},MAE = {-mae:.3f}, MAPE = {-mape:.3f}")

current_r2_score=np.mean(r2_scores_ls)
# scores = cross_val_score(model, X, y_binned, cv=skf, scoring=make_scorer(r2_score))
# print('r2_scores:',scores)
# current_r2_score=scores.mean()
# 只对更优的结果进行保存
# 记录模型的特征重要性
feature_importance_final=model.feature_importances_
importance_dict = dict(zip(feature_importance_final, feature_names))
importance_tuplelist = [(abs(value), feature) for (value, feature) in importance_dict.items()]
importance_sorted = sorted(importance_tuplelist, reverse=True)
x_feature = [j for (i, j) in importance_sorted]
y_importance_value = [i for (i, j) in importance_sorted]
print(f'{model_name}_tree_feature_names(top15):')
for i in x_feature[:15]:
    print(i)
print(f'average_{model_name}_feature_values(top15):')
for i in y_importance_value[:15]:
    print(i)

#记录更优的分数
best_r2_score=current_r2_score
best_mae_score=np.mean(mae_scores)
best_mape_score=np.mean(mape_scores)
if save_model:
    joblib.dump(model,   f'models/{model_name}/{model_name}_optimization2.pkl')  #保存模型
    # print(f'保存更优{model_name}模型文件') 
print('平均 R2 分数:', best_r2_score)
print('平均 MAPE 分数:', best_mape_score * 100)  # 转换为百分比
print('平均MAE分数:',best_mae_score)
    
"""
打印并输出分数至表格
"""

df = pd.read_excel(score_path)
# 添加新数据的示例
new_data = {'Model': f'{model_name}', 'R^2': best_r2_score, 'MAPE': best_mape_score * 100, 'MAE': best_mae_score}
new_data_df = pd.DataFrame([new_data])

# 使用 pd.concat() 合并原 DataFrame 和新的 DataFrame
df = pd.concat([df, new_data_df], ignore_index=True)
# 保存数据框到Excel表格
df.to_excel(score_path, index = False)


    

文件夹models/gboost已存在
The file 'scores_with_best_parameters.xlsx' exists.


,MagpieData minimum Number,MagpieData maximum Number,MagpieData range Number,MagpieData mean Number,MagpieData avg_dev Number,MagpieData mode Number,MagpieData minimum MendeleevNumber,MagpieData maximum MendeleevNumber,MagpieData range MendeleevNumber,MagpieData mean MendeleevNumber,...,frac f valence electrons,Grain Size,Precipitate Distribution,Habit Plane,Calculated Solid Solution,Calculated Grain Boundary,Calculated Yield Strength,屈服强度,抗拉强度 (UTS),追踪编号
0,12,30,18,12.131276,0.247009,12,52,73,21,68.266826,...,0.000000,4.00,0,0,17.669989,194.321983,226.991971,364.0,393.0,No630-6Al-1Zn-0.15Mn -3
1,12,64,52,12.922815,1.811506,12,23,68,45,67.259700,...,0.054984,36.00,3,1,57.602249,48.504397,121.106646,125.0,200.0,No232-Mg-9Gd-1Sm-0.5Zr-2
2,12,64,52,13.780587,3.401587,12,12,69,57,66.376359,...,0.078055,50.00,3,5,94.613373,69.789053,179.402426,200.0,340.0,No16-Mg–14Gd–3Y–1.8Zn–0.5Zr -9
3,12,64,52,13.179217,2.292508,12,12,68,56,66.814372,...,0.052520,1.50,2,1,76.138523,262.520047,353.658570,290.0,280.0,No96-Mg-9.3Gd-2.7Y-0.65Ag-0.52Zr-0.18Ce -4
4,12,30,18,12.112392,0.217496,12,52,73,21,68.118900,...,0.000000,15.00,0,0,11.604694,86.484728,113.089422,148.0,280.0,No460-3Al-1Zn-0.3Mn -22
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
734,12,38,26,12.053936,0.104733,12,8,73,65,68.099036,...,0.000000,130.00,1,1,12.020723,23.509472,50.530195,58.5,220.0,No744-Mg-3Al-0.4Mn-0.05Sr
735,12,64,52,12.937303,1.825661,12,12,69,57,67.051202,...,0.033095,41.77,3,5,61.765688,53.715298,130.480986,223.0,295.0,No226-Mg-6Gd-3Y-1Zn-0.4Zr-0.7Ag-3
736,12,30,18,12.152650,0.281635,12,7,73,66,67.973850,...,0.000000,30.00,3,2,27.156102,71.029741,113.185842,91.1,144.3,No312-Mg-7.6Al-0.5Zn-1Ca-2
737,12,64,52,12.126080,0.237903,12,12,73,61,68.188783,...,0.003189,52.10,1,1,23.809563,45.036179,83.845741,125.0,217.0,No383-Mg-6Al-Zn-0.3Y-0.6Gd


{'alpha': 0.9, 'ccp_alpha': 0.0, 'criterion': 'friedman_mse', 'init': None, 'learning_rate': 0.09999999999999999, 'loss': 'squared_error', 'max_depth': 4, 'max_features': 'sqrt', 'max_leaf_nodes': None, 'min_impurity_decrease': 0.0, 'min_samples_leaf': 1, 'min_samples_split': 2, 'min_weight_fraction_leaf': 0.0, 'n_estimators': 100, 'n_iter_no_change': None, 'random_state': None, 'subsample': 1.0, 'tol': 0.0001, 'validation_fraction': 0.1, 'verbose': 0, 'warm_start': False}
gboost_tree_feature_names(top15):
Calculated Grain Boundary
Grain Size
Calculated Yield Strength
Habit Plane
MagpieData avg_dev NfUnfilled
mean AtomicRadius
MagpieData mean MendeleevNumber
MagpieData avg_dev MendeleevNumber
frac f valence electrons
MagpieData avg_dev GSmagmom
MagpieData avg_dev MeltingT
MagpieData range NUnfilled
mean AtomicWeight
MagpieData mean NfValence
MagpieData mean Row
average_gboost_feature_values(top15):
0.11065438229799714
0.09988030483285641
0.06960306213180764
0.06329296102084081
0.047823

In [13]:
from sklearn.model_selection import GridSearchCV

# 这里采用xgboost算法进行预测
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold,train_test_split,cross_validate,cross_val_score,KFold
from sklearn.metrics import r2_score, mean_absolute_percentage_error,mean_absolute_error,make_scorer
from xgboost import XGBRegressor
from utils import create_empty_ls,create_dir,mycopyfile,mycopydir
import joblib
from openpyxl import load_workbook
import os
"""
路径的配置和选择
"""
model_name='xgboost'

save_model=True
del_0qf=True #是否删除0屈服强度数据
main_path=f'models/{model_name}' # 这里是模型所在的文件位置
create_dir(main_path,is_mainpath=True)
# excel='index/xgboost_index.xlsx'
# 如果分数表格不存在就新建一个
score_path='scores_with_best_parameters.xlsx'
if os.path.exists(score_path):
    print(f"The file '{score_path}' exists.")
else:
    print(f"The file '{score_path}' does not exist.")
    df = pd.DataFrame(columns=['Model', 'R^2', 'MAPE', 'MAE'])
    df.to_excel(score_path, index=False)


"""
文件读取部分
"""
# 读取 Excel 文件
df = pd.read_excel('train_set_new.xlsx', index_col=0)  # 引入这一列之后，原本的第一列就是一个索引，第0列会从有意义的列开始
# 删除包含空值的行
if del_0qf:
    df = df[df['屈服强度'] != 0].reset_index(drop=True)
    # df = df[df['计算晶界'] != 0].reset_index(drop=True)
    # df = df[df['计算固溶'] != 0].reset_index(drop=True)
display(df)
np.random.seed(200) #设置随机数种子
feature_names = df.drop(columns=['Precipitate Distribution',
       '屈服强度', '抗拉强度 (UTS)', '追踪编号']).columns
X = df.drop(columns=['Precipitate Distribution',
       '屈服强度', '抗拉强度 (UTS)', '追踪编号'])  # 特征: 最后四列之前的所有列
y = df.iloc[:, -3]  # 目标: 倒数第四列
# 将目标变量分箱为10个类别
bins = np.linspace(y.min(), y.max(), 11)
y_binned = np.digitize(y, bins)
# print('y_binned',y_binned)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2,random_state=200)

"""
设置初始化
"""
n_splits=5
# 初始化StratifiedKFold
kf = KFold(n_splits=5, shuffle=True, random_state=200)# 注意这里修改了random_state以确保每次初始化不同

# 初始化XGBoost回归器
model = XGBRegressor(random_state=200)# 设置随机种子以确保结果的可重复性
# 初始化列表以存储每个折叠的分数
# 定义评估指标
scoring = {
    'r2':make_scorer(r2_score,greater_is_better=True),
    'MAE': make_scorer(mean_absolute_error, greater_is_better=False),
    'MAPE': make_scorer(mean_absolute_percentage_error, greater_is_better=False)
}
r2_scores_ls = []
mape_scores = []
mae_scores=[]
best_r2_score = float('-inf')
best_mae_score= float('-inf')
best_mape_score= float('-inf')
ls1=['fold'+str(i) for i in range(1,n_splits+1)]
fold_dict=dict(zip(ls1,[0 for i in range(0,len(ls1))]))

# 可修改
parameters={'max_depth':[3],#1,3,5
            'n_estimators':[200],#50,100,150,200
            'learning_rate':np.arange(0.08, 0.16, 0.01), #0.01,0.1,0.2
            'gamma':np.arange(0.08, 0.16, 0.02)} #

# 可修改
model = XGBRegressor(random_state=200)  # 注意这里修改了random_state以确保每次初始化不同
grid_search=GridSearchCV(model,parameters,scoring='r2',cv=kf) # 选择r2作为性能评估参数
grid_search.fit(X_train,y_train)
best_parameter=grid_search.best_estimator_


# 可修改
model = XGBRegressor(n_estimators=best_parameter.n_estimators,gamma=best_parameter.gamma,learning_rate=best_parameter.learning_rate,max_depth=best_parameter.max_depth)  # 注意这里修改了random_state以确保每次初始化不同
model.fit(X_train, y_train)
print(model.get_params())
# 计算分折交叉验证结果
results = cross_validate(model,X,y,cv=kf, scoring=scoring, return_train_score=False)
# 输出分折交叉验证结果
for i, (r2, mae, mape) in enumerate(zip(results['test_r2'],results['test_MAE'], results['test_MAPE']), start=1):
    # 记录不同折的分数结果
    r2_scores_ls.append(r2)
    mape_scores.append(-mape)
    mae_scores.append(-mae)
    # print(f"折 {i}: r2={r2:.3f},MAE = {-mae:.3f}, MAPE = {-mape:.3f}")

current_r2_score=np.mean(r2_scores_ls)
# scores = cross_val_score(model, X, y_binned, cv=skf, scoring=make_scorer(r2_score))
# print('r2_scores:',scores)
# current_r2_score=scores.mean()
# 只对更优的结果进行保存
# 记录模型的特征重要性
feature_importance_final=model.feature_importances_
importance_dict = dict(zip(feature_importance_final, feature_names))
importance_tuplelist = [(abs(value), feature) for (value, feature) in importance_dict.items()]
importance_sorted = sorted(importance_tuplelist, reverse=True)
x_feature = [j for (i, j) in importance_sorted]
y_importance_value = [i for (i, j) in importance_sorted]
print(f'{model_name}_tree_feature_names(top15):')
for i in x_feature[:15]:
    print(i)
print(f'average_{model_name}_feature_values(top15):')
for i in y_importance_value[:15]:
    print(i)

#记录更优的分数
best_r2_score=current_r2_score
best_mae_score=np.mean(mae_scores)
best_mape_score=np.mean(mape_scores)
if save_model:
    joblib.dump(model,   f'models/{model_name}/{model_name}_optimization.pkl')  #保存模型
    # print(f'保存更优{model_name}模型文件') 
print('平均 R2 分数:', best_r2_score)
print('平均 MAPE 分数:', best_mape_score * 100)  # 转换为百分比
print('平均MAE分数:',best_mae_score)


    
"""
打印并输出分数至表格
"""

df = pd.read_excel(score_path)
# 添加新数据的示例
new_data = {'Model': f'{model_name}', 'R^2': best_r2_score, 'MAPE': best_mape_score * 100, 'MAE': best_mae_score}
new_data_df = pd.DataFrame([new_data])

# 使用 pd.concat() 合并原 DataFrame 和新的 DataFrame
df = pd.concat([df, new_data_df], ignore_index=True)
# 保存数据框到Excel表格
df.to_excel(score_path, index = False)


    

文件夹models/xgboost已存在
The file 'scores_with_best_parameters.xlsx' exists.


,MagpieData minimum Number,MagpieData maximum Number,MagpieData range Number,MagpieData mean Number,MagpieData avg_dev Number,MagpieData mode Number,MagpieData minimum MendeleevNumber,MagpieData maximum MendeleevNumber,MagpieData range MendeleevNumber,MagpieData mean MendeleevNumber,...,frac f valence electrons,Grain Size,Precipitate Distribution,Habit Plane,Calculated Solid Solution,Calculated Grain Boundary,Calculated Yield Strength,屈服强度,抗拉强度 (UTS),追踪编号
0,12,30,18,12.131276,0.247009,12,52,73,21,68.266826,...,0.000000,4.00,0,0,17.669989,194.321983,226.991971,364.0,393.0,No630-6Al-1Zn-0.15Mn -3
1,12,64,52,12.922815,1.811506,12,23,68,45,67.259700,...,0.054984,36.00,3,1,57.602249,48.504397,121.106646,125.0,200.0,No232-Mg-9Gd-1Sm-0.5Zr-2
2,12,64,52,13.780587,3.401587,12,12,69,57,66.376359,...,0.078055,50.00,3,5,94.613373,69.789053,179.402426,200.0,340.0,No16-Mg–14Gd–3Y–1.8Zn–0.5Zr -9
3,12,64,52,13.179217,2.292508,12,12,68,56,66.814372,...,0.052520,1.50,2,1,76.138523,262.520047,353.658570,290.0,280.0,No96-Mg-9.3Gd-2.7Y-0.65Ag-0.52Zr-0.18Ce -4
4,12,30,18,12.112392,0.217496,12,52,73,21,68.118900,...,0.000000,15.00,0,0,11.604694,86.484728,113.089422,148.0,280.0,No460-3Al-1Zn-0.3Mn -22
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
734,12,38,26,12.053936,0.104733,12,8,73,65,68.099036,...,0.000000,130.00,1,1,12.020723,23.509472,50.530195,58.5,220.0,No744-Mg-3Al-0.4Mn-0.05Sr
735,12,64,52,12.937303,1.825661,12,12,69,57,67.051202,...,0.033095,41.77,3,5,61.765688,53.715298,130.480986,223.0,295.0,No226-Mg-6Gd-3Y-1Zn-0.4Zr-0.7Ag-3
736,12,30,18,12.152650,0.281635,12,7,73,66,67.973850,...,0.000000,30.00,3,2,27.156102,71.029741,113.185842,91.1,144.3,No312-Mg-7.6Al-0.5Zn-1Ca-2
737,12,64,52,12.126080,0.237903,12,12,73,61,68.188783,...,0.003189,52.10,1,1,23.809563,45.036179,83.845741,125.0,217.0,No383-Mg-6Al-Zn-0.3Y-0.6Gd


{'objective': 'reg:squarederror', 'base_score': None, 'booster': None, 'callbacks': None, 'colsample_bylevel': None, 'colsample_bynode': None, 'colsample_bytree': None, 'device': None, 'early_stopping_rounds': None, 'enable_categorical': False, 'eval_metric': None, 'feature_types': None, 'gamma': 0.08, 'grow_policy': None, 'importance_type': None, 'interaction_constraints': None, 'learning_rate': 0.08, 'max_bin': None, 'max_cat_threshold': None, 'max_cat_to_onehot': None, 'max_delta_step': None, 'max_depth': 3, 'max_leaves': None, 'min_child_weight': None, 'missing': nan, 'monotone_constraints': None, 'multi_strategy': None, 'n_estimators': 200, 'n_jobs': None, 'num_parallel_tree': None, 'random_state': None, 'reg_alpha': None, 'reg_lambda': None, 'sampling_method': None, 'scale_pos_weight': None, 'subsample': None, 'tree_method': None, 'validate_parameters': None, 'verbosity': None}
xgboost_tree_feature_names(top15):
Shear modulus local mismatch
MagpieData range SpaceGroupNumber
frac 

DT`

In [14]:
# from sklearn.model_selection import GridSearchCV
# 
# # 这里采用xgboost算法进行预测
# import numpy as np
# import pandas as pd
# from sklearn.model_selection import StratifiedKFold,train_test_split,cross_validate,cross_val_score,KFold
# from sklearn.metrics import r2_score, mean_absolute_percentage_error,mean_absolute_error,make_scorer
# from xgboost import XGBRegressor
# from sklearn.ensemble import RandomForestRegressor
# from sklearn.ensemble import GradientBoostingRegressor  # 导入GradientBoostingRegressor
# from sklearn.tree import DecisionTreeRegressor 
# from sklearn.inspection import permutation_importance
# from utils import create_empty_ls,create_dir,mycopyfile,mycopydir
# import joblib
# from openpyxl import load_workbook
# import os
# """
# 路径的配置和选择
# """
# model_name='dt'
# 
# save_model=True
# del_0qf=True #是否删除0屈服强度数据
# main_path=f'models/{model_name}' # 这里是模型所在的文件位置
# create_dir(main_path,is_mainpath=True)
# # excel='index/xgboost_index.xlsx'
# # 如果分数表格不存在就新建一个
# score_path='scores_with_best_parameters.xlsx'
# if os.path.exists(score_path):
#     print(f"The file '{score_path}' exists.")
# else:
#     print(f"The file '{score_path}' does not exist.")
#     df = pd.DataFrame(columns=['Model', 'R^2', 'MAPE', 'MAE'])
#     df.to_excel(score_path, index=False)
# 
# 
# """
# 文件读取部分
# """
# # 读取 Excel 文件
# df = pd.read_excel('train_set_new.xlsx', index_col=0)  # 引入这一列之后，原本的第一列就是一个索引，第0列会从有意义的列开始
# # 删除包含空值的行
# if del_0qf:
#     df = df[df['屈服强度'] != 0].reset_index(drop=True)
#     # df = df[df['计算晶界'] != 0].reset_index(drop=True)
#     # df = df[df['计算固溶'] != 0].reset_index(drop=True)
# display(df)
# np.random.seed(200) #设置随机数种子
# feature_names = df.drop(columns=['Precipitate Distribution',
#        '屈服强度', '抗拉强度 (UTS)', '追踪编号']).columns
# X = df.drop(columns=['Precipitate Distribution',
#        '屈服强度', '抗拉强度 (UTS)', '追踪编号'])  # 特征: 最后四列之前的所有列
# y = df.iloc[:, -3]  # 目标: 倒数第四列
# # 将目标变量分箱为10个类别
# bins = np.linspace(y.min(), y.max(), 11)
# y_binned = np.digitize(y, bins)
# # print('y_binned',y_binned)
# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2,random_state=200)
# 
# """
# 设置初始化
# """
# n_splits=5
# # 初始化StratifiedKFold
# kf = KFold(n_splits=5, shuffle=True, random_state=200)# 注意这里修改了random_state以确保每次初始化不同
# 
# # 初始化列表以存储每个折叠的分数
# # 定义评估指标
# scoring = {
#     'r2':make_scorer(r2_score,greater_is_better=True),
#     'MAE': make_scorer(mean_absolute_error, greater_is_better=False),
#     'MAPE': make_scorer(mean_absolute_percentage_error, greater_is_better=False)
# }
# r2_scores_ls = []
# mape_scores = []
# mae_scores=[]
# best_r2_score = float('-inf')
# best_mae_score= float('-inf')
# best_mape_score= float('-inf')
# ls1=['fold'+str(i) for i in range(1,n_splits+1)]
# fold_dict=dict(zip(ls1,[0 for i in range(0,len(ls1))]))
# 
# # 可修改
# parameters={'max_depth':[n for n in range(3,8,2)], # 最大深度5,7,9
#             # 'n_estimators':[50*n for n in range(1,5)],# DT个数 50,100,150,200
#             'min_samples_leaf':[1,2,'None'], # 叶子结点所需最小样本数
#             'min_samples_split':[2,3,4,'None'],#分裂最小样本数
#             'max_features':['auto','sqrt','log2','None'],
#             'criterion':['absolute_error', 'squared_error', 'friedman_mse', 'poisson'] 
#             } #最大特征数量
# 
# # 可修改
# model = DecisionTreeRegressor(random_state=200)
# # 注意这里修改了random_state以确保每次初始化不同
# grid_search=GridSearchCV(model,parameters,scoring='r2',cv=kf) # 选择r2作为性能评估参数
# grid_search.fit(X_train,y_train)
# best_parameter=grid_search.best_estimator_
# 
# 
# # 可修改
# model = DecisionTreeRegressor(min_samples_leaf=best_parameter.min_samples_leaf,min_samples_split=best_parameter.min_samples_split,max_depth=best_parameter.max_depth,max_features=best_parameter.max_features,criterion=best_parameter.criterion)  # 注意这里修改了random_state以确保每次初始化不同
# model.fit(X_train, y_train)
# print(model.get_params())
# # 计算分折交叉验证结果
# results = cross_validate(model,X,y,cv=kf, scoring=scoring, return_train_score=False)
# # 输出分折交叉验证结果
# for i, (r2, mae, mape) in enumerate(zip(results['test_r2'],results['test_MAE'], results['test_MAPE']), start=1):
#     # 记录不同折的分数结果
#     r2_scores_ls.append(r2)
#     mape_scores.append(-mape)
#     mae_scores.append(-mae)
#     # print(f"折 {i}: r2={r2:.3f},MAE = {-mae:.3f}, MAPE = {-mape:.3f}")
# 
# current_r2_score=np.mean(r2_scores_ls)
# # scores = cross_val_score(model, X, y_binned, cv=skf, scoring=make_scorer(r2_score))
# # print('r2_scores:',scores)
# # current_r2_score=scores.mean()
# # 只对更优的结果进行保存
# # 记录模型的特征重要性
# feature_importance_final=model.feature_importances_
# importance_dict = dict(zip(feature_importance_final, feature_names))
# importance_tuplelist = [(abs(value), feature) for (value, feature) in importance_dict.items()]
# importance_sorted = sorted(importance_tuplelist, reverse=True)
# x_feature = [j for (i, j) in importance_sorted]
# y_importance_value = [i for (i, j) in importance_sorted]
# print(f'{model_name}_tree_feature_names(top15):')
# for i in x_feature[:15]:
#     print(i)
# print(f'average_{model_name}_feature_values(top15):')
# for i in y_importance_value[:15]:
#     print(i)
# 
# #记录更优的分数
# best_r2_score=current_r2_score
# best_mae_score=np.mean(mae_scores)
# best_mape_score=np.mean(mape_scores)
# if save_model:
#     joblib.dump(model,   f'models/{model_name}/{model_name}_optimization.pkl')  #保存模型
#     # print(f'保存更优{model_name}模型文件') 
# print('平均 R2 分数:', best_r2_score)
# print('平均 MAPE 分数:', best_mape_score * 100)  # 转换为百分比
# print('平均MAE分数:',best_mae_score)
# 
# 
#     
# """
# 打印并输出分数至表格
# """
# 
# df = pd.read_excel(score_path)
# # 添加新数据的示例
# new_data = {'Model': f'{model_name}', 'R^2': best_r2_score, 'MAPE': best_mape_score * 100, 'MAE': best_mae_score}
# new_data_df = pd.DataFrame([new_data])
# 
# # 使用 pd.concat() 合并原 DataFrame 和新的 DataFrame
# df = pd.concat([df, new_data_df], ignore_index=True)
# # 保存数据框到Excel表格
# df.to_excel(score_path, index = False)
# 
# 
#     

In [15]:
# model = DecisionTreeRegressor(random_state=200)  # 注意这里修改了random_state以确保每次初始化不同
# model.fit(X_train, y_train)
# # 计算分折交叉验证结果
# results = cross_validate(model,X,y,cv=kf, scoring=scoring, return_train_score=False)
# # 输出分折交叉验证结果
# for i, (r2, mae, mape) in enumerate(zip(results['test_r2'],results['test_MAE'], results['test_MAPE']), start=1):
#     # 记录不同折的分数结果
#     r2_scores_ls.append(r2)
#     mape_scores.append(-mape)
#     mae_scores.append(-mae)
#     # print(f"折 {i}: r2={r2:.3f},MAE = {-mae:.3f}, MAPE = {-mape:.3f}")
# 
# current_r2_score=np.mean(r2_scores_ls)
# # scores = cross_val_score(model, X, y_binned, cv=skf, scoring=make_scorer(r2_score))
# # print('r2_scores:',scores)
# # current_r2_score=scores.mean()
# # 只对更优的结果进行保存
# # 记录模型的特征重要性
# feature_importance_final=model.feature_importances_
# importance_dict = dict(zip(feature_importance_final, feature_names))
# importance_tuplelist = [(abs(value), feature) for (value, feature) in importance_dict.items()]
# importance_sorted = sorted(importance_tuplelist, reverse=True)
# x_feature = [j for (i, j) in importance_sorted]
# y_importance_value = [i for (i, j) in importance_sorted]
# print(f'{model_name}_tree_feature_names(top15):-------+')
# for i in x_feature[:15]:
#     print(i)
# print(f'average_{model_name}_feature_values(top15):')
# for i in y_importance_value[:15]:
#     print(i)
# #记录更优的分数
# best_r2_score=current_r2_score
# best_mae_score=np.mean(mae_scores)
# best_mape_score=np.mean(mape_scores)
# 
# print('平均 R2 分数:', best_r2_score)
# print('平均 MAPE 分数:', best_mape_score * 100)  # 转换为百分比
# print('平均MAE分数:',best_mae_score)
# model.get_params()

In [16]:
# joblib.dump(model,'models/dt/dt_best.pkl')

In [17]:
# from sklearn.model_selection import GridSearchCV
# 
# # 这里采用xgboost算法进行预测
# import numpy as np
# import pandas as pd
# from sklearn.model_selection import StratifiedKFold,train_test_split,cross_validate,cross_val_score,KFold
# from sklearn.metrics import r2_score, mean_absolute_percentage_error,mean_absolute_error,make_scorer
# from xgboost import XGBRegressor
# from sklearn.ensemble import RandomForestRegressor
# from sklearn.ensemble import GradientBoostingRegressor  # 导入GradientBoostingRegressor
# from sklearn.tree import DecisionTreeRegressor 
# from sklearn.inspection import permutation_importance
# from utils import create_empty_ls,create_dir,mycopyfile,mycopydir
# import joblib
# from openpyxl import load_workbook
# import os
# """
# 路径的配置和选择
# """
# model_name='dt'
# 
# save_model=True
# del_0qf=True #是否删除0屈服强度数据
# main_path=f'models/{model_name}' # 这里是模型所在的文件位置
# create_dir(main_path,is_mainpath=True)
# # excel='index/xgboost_index.xlsx'
# # 如果分数表格不存在就新建一个
# score_path='scores_with_best_parameters.xlsx'
# if os.path.exists(score_path):
#     print(f"The file '{score_path}' exists.")
# else:
#     print(f"The file '{score_path}' does not exist.")
#     df = pd.DataFrame(columns=['Model', 'R^2', 'MAPE', 'MAE'])
#     df.to_excel(score_path, index=False)
# 
# 
# """
# 文件读取部分
# """
# # 读取 Excel 文件
# df = pd.read_excel('train_set_new.xlsx', index_col=0)  # 引入这一列之后，原本的第一列就是一个索引，第0列会从有意义的列开始
# # 删除包含空值的行
# if del_0qf:
#     df = df[df['屈服强度'] != 0].reset_index(drop=True)
#     # df = df[df['计算晶界'] != 0].reset_index(drop=True)
#     # df = df[df['计算固溶'] != 0].reset_index(drop=True)
# display(df)
# np.random.seed(200) #设置随机数种子
# feature_names = df.drop(columns=['Precipitate Distribution',
#        '屈服强度', '抗拉强度 (UTS)', '追踪编号']).columns
# X = df.drop(columns=['Precipitate Distribution',
#        '屈服强度', '抗拉强度 (UTS)', '追踪编号'])  # 特征: 最后四列之前的所有列
# y = df.iloc[:, -3]  # 目标: 倒数第四列
# # 将目标变量分箱为10个类别
# bins = np.linspace(y.min(), y.max(), 11)
# y_binned = np.digitize(y, bins)
# # print('y_binned',y_binned)
# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2,random_state=200)
# 
# """
# 设置初始化
# """
# n_splits=5
# # 初始化StratifiedKFold
# kf = KFold(n_splits=5, shuffle=True, random_state=200)# 注意这里修改了random_state以确保每次初始化不同
# 
# # 初始化列表以存储每个折叠的分数
# # 定义评估指标
# scoring = {
#     'r2':make_scorer(r2_score,greater_is_better=True),
#     'MAE': make_scorer(mean_absolute_error, greater_is_better=False),
#     'MAPE': make_scorer(mean_absolute_percentage_error, greater_is_better=False)
# }
# r2_scores_ls = []
# mape_scores = []
# mae_scores=[]
# best_r2_score = float('-inf')
# best_mae_score= float('-inf')
# best_mape_score= float('-inf')
# ls1=['fold'+str(i) for i in range(1,n_splits+1)]
# fold_dict=dict(zip(ls1,[0 for i in range(0,len(ls1))]))
# 
# # 可修改
# parameters={'max_depth':np.arange(3,6,1), # 最大深度5,7,9
#             # 'n_estimators':[50*n for n in range(1,5)],# DT个数 50,100,150,200
#             'min_samples_leaf':[1], # 叶子结点所需最小样本数
#             'min_samples_split':[2],#分裂最小样本数
#             'max_features':['sqrt','log2'],
#             'criterion':[ 'squared_error'] 
#             } #最大特征数量
# 
# # 可修改
# model = DecisionTreeRegressor(random_state=200)
# # 注意这里修改了random_state以确保每次初始化不同
# grid_search=GridSearchCV(model,parameters,scoring='r2',cv=kf) # 选择r2作为性能评估参数
# grid_search.fit(X_train,y_train)
# best_parameter=grid_search.best_estimator_
# 
# 
# # 可修改
# model = DecisionTreeRegressor(min_samples_leaf=best_parameter.min_samples_leaf,min_samples_split=best_parameter.min_samples_split,max_depth=best_parameter.max_depth,max_features=best_parameter.max_features,criterion=best_parameter.criterion)  # 注意这里修改了random_state以确保每次初始化不同
# model.fit(X_train, y_train)
# print(model.get_params())
# # 计算分折交叉验证结果
# results = cross_validate(model,X,y,cv=kf, scoring=scoring, return_train_score=False)
# # 输出分折交叉验证结果
# for i, (r2, mae, mape) in enumerate(zip(results['test_r2'],results['test_MAE'], results['test_MAPE']), start=1):
#     # 记录不同折的分数结果
#     r2_scores_ls.append(r2)
#     mape_scores.append(-mape)
#     mae_scores.append(-mae)
#     # print(f"折 {i}: r2={r2:.3f},MAE = {-mae:.3f}, MAPE = {-mape:.3f}")
# 
# current_r2_score=np.mean(r2_scores_ls)
# # scores = cross_val_score(model, X, y_binned, cv=skf, scoring=make_scorer(r2_score))
# # print('r2_scores:',scores)
# # current_r2_score=scores.mean()
# # 只对更优的结果进行保存
# # 记录模型的特征重要性
# feature_importance_final=model.feature_importances_
# importance_dict = dict(zip(feature_importance_final, feature_names))
# importance_tuplelist = [(abs(value), feature) for (value, feature) in importance_dict.items()]
# importance_sorted = sorted(importance_tuplelist, reverse=True)
# x_feature = [j for (i, j) in importance_sorted]
# y_importance_value = [i for (i, j) in importance_sorted]
# print(f'{model_name}_tree_feature_names(top15):')
# for i in x_feature[:15]:
#     print(i)
# print(f'average_{model_name}_feature_values(top15):')
# for i in y_importance_value[:15]:
#     print(i)
# 
# #记录更优的分数
# best_r2_score=current_r2_score
# best_mae_score=np.mean(mae_scores)
# best_mape_score=np.mean(mape_scores)
# if save_model:
#     joblib.dump(model,   f'models/{model_name}/{model_name}_optimization.pkl')  #保存模型
#     # print(f'保存更优{model_name}模型文件') 
# print('平均 R2 分数:', best_r2_score)
# print('平均 MAPE 分数:', best_mape_score * 100)  # 转换为百分比
# print('平均MAE分数:',best_mae_score)
# 
# 
#     
# """
# 打印并输出分数至表格
# """
# 
# df = pd.read_excel(score_path)
# # 添加新数据的示例
# new_data = {'Model': f'{model_name}', 'R^2': best_r2_score, 'MAPE': best_mape_score * 100, 'MAE': best_mae_score}
# new_data_df = pd.DataFrame([new_data])
# 
# # 使用 pd.concat() 合并原 DataFrame 和新的 DataFrame
# df = pd.concat([df, new_data_df], ignore_index=True)
# # 保存数据框到Excel表格
# df.to_excel(score_path, index = False)
# 
# 
#     

AdaBoost

In [18]:
from sklearn.model_selection import GridSearchCV

# 这里采用xgboost算法进行预测
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold,train_test_split,cross_validate,cross_val_score,KFold
from sklearn.metrics import r2_score, mean_absolute_percentage_error,mean_absolute_error,make_scorer
from xgboost import XGBRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import GradientBoostingRegressor  # 导入GradientBoostingRegressor
from sklearn.tree import DecisionTreeRegressor 
from sklearn.ensemble import AdaBoostRegressor
from sklearn.inspection import permutation_importance
from utils import create_empty_ls,create_dir,mycopyfile,mycopydir
import joblib
from openpyxl import load_workbook
import os
"""
路径的配置和选择
"""
model_name='adaboost'

save_model=True
del_0qf=True #是否删除0屈服强度数据
main_path=f'models/{model_name}' # 这里是模型所在的文件位置
create_dir(main_path,is_mainpath=True)
# excel='index/xgboost_index.xlsx'
# 如果分数表格不存在就新建一个
score_path='scores_with_best_parameters.xlsx'
if os.path.exists(score_path):
    print(f"The file '{score_path}' exists.")
else:
    print(f"The file '{score_path}' does not exist.")
    df = pd.DataFrame(columns=['Model', 'R^2', 'MAPE', 'MAE'])
    df.to_excel(score_path, index=False)


"""
文件读取部分
"""
# 读取 Excel 文件
df = pd.read_excel('train_set_new.xlsx', index_col=0)  # 引入这一列之后，原本的第一列就是一个索引，第0列会从有意义的列开始
# 删除包含空值的行
if del_0qf:
    df = df[df['屈服强度'] != 0].reset_index(drop=True)
    # df = df[df['计算晶界'] != 0].reset_index(drop=True)
    # df = df[df['计算固溶'] != 0].reset_index(drop=True)
display(df)
np.random.seed(200) #设置随机数种子
feature_names = df.drop(columns=['Precipitate Distribution',
       '屈服强度', '抗拉强度 (UTS)', '追踪编号']).columns
X = df.drop(columns=['Precipitate Distribution',
       '屈服强度', '抗拉强度 (UTS)', '追踪编号'])  # 特征: 最后四列之前的所有列
y = df.iloc[:, -3]  # 目标: 倒数第四列
# 将目标变量分箱为10个类别
bins = np.linspace(y.min(), y.max(), 11)
y_binned = np.digitize(y, bins)
# print('y_binned',y_binned)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2,random_state=200)

"""
设置初始化
"""
n_splits=5
# 初始化StratifiedKFold
kf = KFold(n_splits=5, shuffle=True, random_state=200)# 注意这里修改了random_state以确保每次初始化不同

# 初始化列表以存储每个折叠的分数
# 定义评估指标
scoring = {
    'r2':make_scorer(r2_score,greater_is_better=True),
    'MAE': make_scorer(mean_absolute_error, greater_is_better=False),
    'MAPE': make_scorer(mean_absolute_percentage_error, greater_is_better=False)
}
r2_scores_ls = []
mape_scores = []
mae_scores=[]
best_r2_score = float('-inf')
best_mae_score= float('-inf')
best_mape_score= float('-inf')
ls1=['fold'+str(i) for i in range(1,n_splits+1)]
fold_dict=dict(zip(ls1,[0 for i in range(0,len(ls1))]))

# 可修改
parameters={#'max_depth':[n for n in range(3,8,2)], # 最大深度5,7,9
            'n_estimators':[50*n for n in range(1,5)],# DT个数 50,100,150,200
            # 'min_samples_leaf':[1,2,'None'], # 叶子结点所需最小样本数
            # 'min_samples_split':[2,3,4,'None'],#分裂最小样本数
            'learning_rate':[0.01,0.05,0.1,0.15,0.2], 
            # 'max_features':['auto','sqrt','log2','None'],
            'loss':['square', 'linear', 'exponential'] 
            } #最大特征数量

# 可修改
model = AdaBoostRegressor(random_state=200)
# 注意这里修改了random_state以确保每次初始化不同
grid_search=GridSearchCV(model,parameters,scoring='r2',cv=kf) # 选择r2作为性能评估参数
grid_search.fit(X_train,y_train)
best_parameter=grid_search.best_estimator_


# 可修改
model = AdaBoostRegressor(n_estimators=best_parameter.n_estimators,learning_rate=best_parameter.learning_rate,loss=best_parameter.loss)  # 注意这里修改了random_state以确保每次初始化不同
model.fit(X_train, y_train)
print(model.get_params())
# 计算分折交叉验证结果
results = cross_validate(model,X,y,cv=kf, scoring=scoring, return_train_score=False)
# 输出分折交叉验证结果
for i, (r2, mae, mape) in enumerate(zip(results['test_r2'],results['test_MAE'], results['test_MAPE']), start=1):
    # 记录不同折的分数结果
    r2_scores_ls.append(r2)
    mape_scores.append(-mape)
    mae_scores.append(-mae)
    # print(f"折 {i}: r2={r2:.3f},MAE = {-mae:.3f}, MAPE = {-mape:.3f}")

current_r2_score=np.mean(r2_scores_ls)
# scores = cross_val_score(model, X, y_binned, cv=skf, scoring=make_scorer(r2_score))
# print('r2_scores:',scores)
# current_r2_score=scores.mean()
# 只对更优的结果进行保存
# 记录模型的特征重要性
feature_importance_final=model.feature_importances_
importance_dict = dict(zip(feature_importance_final, feature_names))
importance_tuplelist = [(abs(value), feature) for (value, feature) in importance_dict.items()]
importance_sorted = sorted(importance_tuplelist, reverse=True)
x_feature = [j for (i, j) in importance_sorted]
y_importance_value = [i for (i, j) in importance_sorted]
print(f'{model_name}_tree_feature_names(top15):')
for i in x_feature[:15]:
    print(i)
print(f'average_{model_name}_feature_values(top15):')
for i in y_importance_value[:15]:
    print(i)

#记录更优的分数
best_r2_score=current_r2_score
best_mae_score=np.mean(mae_scores)
best_mape_score=np.mean(mape_scores)
if save_model:
    joblib.dump(model,   f'models/{model_name}/{model_name}_optimization1.pkl')  #保存模型
    # print(f'保存更优{model_name}模型文件') 
print('平均 R2 分数:', best_r2_score)
print('平均 MAPE 分数:', best_mape_score * 100)  # 转换为百分比
print('平均MAE分数:',best_mae_score)


    
"""
打印并输出分数至表格
"""

df = pd.read_excel(score_path)
# 添加新数据的示例
new_data = {'Model': f'{model_name}', 'R^2': best_r2_score, 'MAPE': best_mape_score * 100, 'MAE': best_mae_score}
new_data_df = pd.DataFrame([new_data])

# 使用 pd.concat() 合并原 DataFrame 和新的 DataFrame
df = pd.concat([df, new_data_df], ignore_index=True)
# 保存数据框到Excel表格
df.to_excel(score_path, index = False)


    

文件夹models/adaboost已存在
The file 'scores_with_best_parameters.xlsx' exists.


,MagpieData minimum Number,MagpieData maximum Number,MagpieData range Number,MagpieData mean Number,MagpieData avg_dev Number,MagpieData mode Number,MagpieData minimum MendeleevNumber,MagpieData maximum MendeleevNumber,MagpieData range MendeleevNumber,MagpieData mean MendeleevNumber,...,frac f valence electrons,Grain Size,Precipitate Distribution,Habit Plane,Calculated Solid Solution,Calculated Grain Boundary,Calculated Yield Strength,屈服强度,抗拉强度 (UTS),追踪编号
0,12,30,18,12.131276,0.247009,12,52,73,21,68.266826,...,0.000000,4.00,0,0,17.669989,194.321983,226.991971,364.0,393.0,No630-6Al-1Zn-0.15Mn -3
1,12,64,52,12.922815,1.811506,12,23,68,45,67.259700,...,0.054984,36.00,3,1,57.602249,48.504397,121.106646,125.0,200.0,No232-Mg-9Gd-1Sm-0.5Zr-2
2,12,64,52,13.780587,3.401587,12,12,69,57,66.376359,...,0.078055,50.00,3,5,94.613373,69.789053,179.402426,200.0,340.0,No16-Mg–14Gd–3Y–1.8Zn–0.5Zr -9
3,12,64,52,13.179217,2.292508,12,12,68,56,66.814372,...,0.052520,1.50,2,1,76.138523,262.520047,353.658570,290.0,280.0,No96-Mg-9.3Gd-2.7Y-0.65Ag-0.52Zr-0.18Ce -4
4,12,30,18,12.112392,0.217496,12,52,73,21,68.118900,...,0.000000,15.00,0,0,11.604694,86.484728,113.089422,148.0,280.0,No460-3Al-1Zn-0.3Mn -22
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
734,12,38,26,12.053936,0.104733,12,8,73,65,68.099036,...,0.000000,130.00,1,1,12.020723,23.509472,50.530195,58.5,220.0,No744-Mg-3Al-0.4Mn-0.05Sr
735,12,64,52,12.937303,1.825661,12,12,69,57,67.051202,...,0.033095,41.77,3,5,61.765688,53.715298,130.480986,223.0,295.0,No226-Mg-6Gd-3Y-1Zn-0.4Zr-0.7Ag-3
736,12,30,18,12.152650,0.281635,12,7,73,66,67.973850,...,0.000000,30.00,3,2,27.156102,71.029741,113.185842,91.1,144.3,No312-Mg-7.6Al-0.5Zn-1Ca-2
737,12,64,52,12.126080,0.237903,12,12,73,61,68.188783,...,0.003189,52.10,1,1,23.809563,45.036179,83.845741,125.0,217.0,No383-Mg-6Al-Zn-0.3Y-0.6Gd


{'estimator': None, 'learning_rate': 0.15, 'loss': 'exponential', 'n_estimators': 50, 'random_state': None}
adaboost_tree_feature_names(top15):
Calculated Yield Strength
Grain Size
Habit Plane
frac f valence electrons
Interant d electrons
Lambda entropy
Gd fraction
Interant electrons
MagpieData mean GSvolume_pa
Radii gamma
MagpieData avg_dev Electronegativity
Calculated Grain Boundary
Zr fraction
MagpieData avg_dev MeltingT
Yang omega
average_adaboost_feature_values(top15):
0.377446570445821
0.1341845608848629
0.08848916988253257
0.03769361261344172
0.03275647789791118
0.030473669635562865
0.02447079738300413
0.018805339775549188
0.0183574964773228
0.018209693888214685
0.01495300826805707
0.012760750850656172
0.011933253664887216
0.01131924803374629
0.010921008207022275
平均 R2 分数: 0.5867462122668904
平均 MAPE 分数: 29.68279735318527
平均MAE分数: 41.874776992225804


In [19]:
best_parameter

AdaBoostRegressor(learning_rate=0.15, loss='exponential', random_state=200)

In [20]:
model = AdaBoostRegressor(random_state=200)  # 注意这里修改了random_state以确保每次初始化不同
model.fit(X_train, y_train)
# 计算分折交叉验证结果
results = cross_validate(model,X,y,cv=kf, scoring=scoring, return_train_score=False)
# 输出分折交叉验证结果
for i, (r2, mae, mape) in enumerate(zip(results['test_r2'],results['test_MAE'], results['test_MAPE']), start=1):
    # 记录不同折的分数结果
    r2_scores_ls.append(r2)
    mape_scores.append(-mape)
    mae_scores.append(-mae)
    # print(f"折 {i}: r2={r2:.3f},MAE = {-mae:.3f}, MAPE = {-mape:.3f}")

current_r2_score=np.mean(r2_scores_ls)
# scores = cross_val_score(model, X, y_binned, cv=skf, scoring=make_scorer(r2_score))
# print('r2_scores:',scores)
# current_r2_score=scores.mean()
# 只对更优的结果进行保存
# 记录模型的特征重要性
feature_importance_final=model.feature_importances_
importance_dict = dict(zip(feature_importance_final, feature_names))
importance_tuplelist = [(abs(value), feature) for (value, feature) in importance_dict.items()]
importance_sorted = sorted(importance_tuplelist, reverse=True)
x_feature = [j for (i, j) in importance_sorted]
y_importance_value = [i for (i, j) in importance_sorted]
print(f'{model_name}_tree_feature_names(top15):-------+')
for i in x_feature[:15]:
    print(i)
print(f'average_{model_name}_feature_values(top15):')
for i in y_importance_value[:15]:
    print(i)
#记录更优的分数
best_r2_score=current_r2_score
best_mae_score=np.mean(mae_scores)
best_mape_score=np.mean(mape_scores)

print('平均 R2 分数:', best_r2_score)
print('平均 MAPE 分数:', best_mape_score * 100)  # 转换为百分比
print('平均MAE分数:',best_mae_score)
model.get_params()

adaboost_tree_feature_names(top15):-------+
Calculated Yield Strength
Grain Size
Habit Plane
Calculated Grain Boundary
Lambda entropy
Interant d electrons
MagpieData avg_dev GSvolume_pa
MagpieData mean NUnfilled
MagpieData mean NValence
MagpieData avg_dev NUnfilled
Mn fraction
avg d valence electrons
Interant electrons
MagpieData avg_dev MeltingT
Mean cohesive energy
average_adaboost_feature_values(top15):
0.26017919321502
0.1353098474314048
0.13290582622506625
0.08878414612113379
0.05035822034444781
0.030320635983841288
0.0285178868112548
0.025180350195334277
0.01985641435552712
0.016165659861332587
0.013774303026067624
0.01291063153806397
0.012574729089485748
0.011957632778459277
0.010271261309089733
平均 R2 分数: 0.5679486552154411
平均 MAPE 分数: 31.5604737503363
平均MAE分数: 43.591819653582235


{'estimator': None,
 'learning_rate': 1.0,
 'loss': 'linear',
 'n_estimators': 50,
 'random_state': 200}

Adaboost再调参

In [21]:
from sklearn.model_selection import GridSearchCV

# 这里采用xgboost算法进行预测
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold,train_test_split,cross_validate,cross_val_score,KFold
from sklearn.metrics import r2_score, mean_absolute_percentage_error,mean_absolute_error,make_scorer
from xgboost import XGBRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import GradientBoostingRegressor  # 导入GradientBoostingRegressor
from sklearn.tree import DecisionTreeRegressor 
from sklearn.ensemble import AdaBoostRegressor
from sklearn.inspection import permutation_importance
from utils import create_empty_ls,create_dir,mycopyfile,mycopydir
import joblib
from openpyxl import load_workbook
import os
"""
路径的配置和选择
"""
model_name='adaboost'

save_model=True
del_0qf=True #是否删除0屈服强度数据
main_path=f'models/{model_name}' # 这里是模型所在的文件位置
create_dir(main_path,is_mainpath=True)
# excel='index/xgboost_index.xlsx'
# 如果分数表格不存在就新建一个
score_path='scores_with_best_parameters.xlsx'
if os.path.exists(score_path):
    print(f"The file '{score_path}' exists.")
else:
    print(f"The file '{score_path}' does not exist.")
    df = pd.DataFrame(columns=['Model', 'R^2', 'MAPE', 'MAE'])
    df.to_excel(score_path, index=False)


"""
文件读取部分
"""
# 读取 Excel 文件
df = pd.read_excel('train_set_new.xlsx', index_col=0)  # 引入这一列之后，原本的第一列就是一个索引，第0列会从有意义的列开始
# 删除包含空值的行
if del_0qf:
    df = df[df['屈服强度'] != 0].reset_index(drop=True)
    # df = df[df['计算晶界'] != 0].reset_index(drop=True)
    # df = df[df['计算固溶'] != 0].reset_index(drop=True)
display(df)
np.random.seed(200) #设置随机数种子
feature_names = df.drop(columns=['Precipitate Distribution',
       '屈服强度', '抗拉强度 (UTS)', '追踪编号']).columns
X = df.drop(columns=['Precipitate Distribution',
       '屈服强度', '抗拉强度 (UTS)', '追踪编号'])  # 特征: 最后四列之前的所有列
y = df.iloc[:, -3]  # 目标: 倒数第四列
# 将目标变量分箱为10个类别
bins = np.linspace(y.min(), y.max(), 11)
y_binned = np.digitize(y, bins)
# print('y_binned',y_binned)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2,random_state=200)

"""
设置初始化
"""
n_splits=5
# 初始化StratifiedKFold
kf = KFold(n_splits=5, shuffle=True, random_state=200)# 注意这里修改了random_state以确保每次初始化不同

# 初始化列表以存储每个折叠的分数
# 定义评估指标
scoring = {
    'r2':make_scorer(r2_score,greater_is_better=True),
    'MAE': make_scorer(mean_absolute_error, greater_is_better=False),
    'MAPE': make_scorer(mean_absolute_percentage_error, greater_is_better=False)
}
r2_scores_ls = []
mape_scores = []
mae_scores=[]
best_r2_score = float('-inf')
best_mae_score= float('-inf')
best_mape_score= float('-inf')
ls1=['fold'+str(i) for i in range(1,n_splits+1)]
fold_dict=dict(zip(ls1,[0 for i in range(0,len(ls1))]))

# 可修改
parameters={#'max_depth':[n for n in range(3,8,2)], # 最大深度5,7,9
            'n_estimators':[25*n for n in range(2,4)],# DT个数 50,100,150,200
            # 'min_samples_leaf':[1,2,'None'], # 叶子结点所需最小样本数
            # 'min_samples_split':[2,3,4,'None'],#分裂最小样本数
            'learning_rate':np.arange(0.13,0.2,0.02), 
            # 'max_features':['auto','sqrt','log2','None'],
            'loss':['exponential'] 
            } #最大特征数量

# 可修改
model = AdaBoostRegressor(random_state=200)
# 注意这里修改了random_state以确保每次初始化不同
grid_search=GridSearchCV(model,parameters,scoring='r2',cv=kf) # 选择r2作为性能评估参数
grid_search.fit(X_train,y_train)
best_parameter=grid_search.best_estimator_


# 可修改
model = AdaBoostRegressor(n_estimators=best_parameter.n_estimators,learning_rate=best_parameter.learning_rate,loss=best_parameter.loss)  # 注意这里修改了random_state以确保每次初始化不同
model.fit(X_train, y_train)
print(model.get_params())
# 计算分折交叉验证结果
results = cross_validate(model,X,y,cv=kf, scoring=scoring, return_train_score=False)
# 输出分折交叉验证结果
for i, (r2, mae, mape) in enumerate(zip(results['test_r2'],results['test_MAE'], results['test_MAPE']), start=1):
    # 记录不同折的分数结果
    r2_scores_ls.append(r2)
    mape_scores.append(-mape)
    mae_scores.append(-mae)
    # print(f"折 {i}: r2={r2:.3f},MAE = {-mae:.3f}, MAPE = {-mape:.3f}")

current_r2_score=np.mean(r2_scores_ls)
# scores = cross_val_score(model, X, y_binned, cv=skf, scoring=make_scorer(r2_score))
# print('r2_scores:',scores)
# current_r2_score=scores.mean()
# 只对更优的结果进行保存
# 记录模型的特征重要性
feature_importance_final=model.feature_importances_
importance_dict = dict(zip(feature_importance_final, feature_names))
importance_tuplelist = [(abs(value), feature) for (value, feature) in importance_dict.items()]
importance_sorted = sorted(importance_tuplelist, reverse=True)
x_feature = [j for (i, j) in importance_sorted]
y_importance_value = [i for (i, j) in importance_sorted]
print(f'{model_name}_tree_feature_names(top15):')
for i in x_feature[:15]:
    print(i)
print(f'average_{model_name}_feature_values(top15):')
for i in y_importance_value[:15]:
    print(i)

#记录更优的分数
best_r2_score=current_r2_score
best_mae_score=np.mean(mae_scores)
best_mape_score=np.mean(mape_scores)
if save_model:
    joblib.dump(model,   f'models/{model_name}/{model_name}_optimization2.pkl')  #保存模型
    # print(f'保存更优{model_name}模型文件') 
print('平均 R2 分数:', best_r2_score)
print('平均 MAPE 分数:', best_mape_score * 100)  # 转换为百分比
print('平均MAE分数:',best_mae_score)


    
"""
打印并输出分数至表格
"""

df = pd.read_excel(score_path)
# 添加新数据的示例
new_data = {'Model': f'{model_name}', 'R^2': best_r2_score, 'MAPE': best_mape_score * 100, 'MAE': best_mae_score}
new_data_df = pd.DataFrame([new_data])

# 使用 pd.concat() 合并原 DataFrame 和新的 DataFrame
df = pd.concat([df, new_data_df], ignore_index=True)
# 保存数据框到Excel表格
df.to_excel(score_path, index = False)

    

文件夹models/adaboost已存在
The file 'scores_with_best_parameters.xlsx' exists.


,MagpieData minimum Number,MagpieData maximum Number,MagpieData range Number,MagpieData mean Number,MagpieData avg_dev Number,MagpieData mode Number,MagpieData minimum MendeleevNumber,MagpieData maximum MendeleevNumber,MagpieData range MendeleevNumber,MagpieData mean MendeleevNumber,...,frac f valence electrons,Grain Size,Precipitate Distribution,Habit Plane,Calculated Solid Solution,Calculated Grain Boundary,Calculated Yield Strength,屈服强度,抗拉强度 (UTS),追踪编号
0,12,30,18,12.131276,0.247009,12,52,73,21,68.266826,...,0.000000,4.00,0,0,17.669989,194.321983,226.991971,364.0,393.0,No630-6Al-1Zn-0.15Mn -3
1,12,64,52,12.922815,1.811506,12,23,68,45,67.259700,...,0.054984,36.00,3,1,57.602249,48.504397,121.106646,125.0,200.0,No232-Mg-9Gd-1Sm-0.5Zr-2
2,12,64,52,13.780587,3.401587,12,12,69,57,66.376359,...,0.078055,50.00,3,5,94.613373,69.789053,179.402426,200.0,340.0,No16-Mg–14Gd–3Y–1.8Zn–0.5Zr -9
3,12,64,52,13.179217,2.292508,12,12,68,56,66.814372,...,0.052520,1.50,2,1,76.138523,262.520047,353.658570,290.0,280.0,No96-Mg-9.3Gd-2.7Y-0.65Ag-0.52Zr-0.18Ce -4
4,12,30,18,12.112392,0.217496,12,52,73,21,68.118900,...,0.000000,15.00,0,0,11.604694,86.484728,113.089422,148.0,280.0,No460-3Al-1Zn-0.3Mn -22
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
734,12,38,26,12.053936,0.104733,12,8,73,65,68.099036,...,0.000000,130.00,1,1,12.020723,23.509472,50.530195,58.5,220.0,No744-Mg-3Al-0.4Mn-0.05Sr
735,12,64,52,12.937303,1.825661,12,12,69,57,67.051202,...,0.033095,41.77,3,5,61.765688,53.715298,130.480986,223.0,295.0,No226-Mg-6Gd-3Y-1Zn-0.4Zr-0.7Ag-3
736,12,30,18,12.152650,0.281635,12,7,73,66,67.973850,...,0.000000,30.00,3,2,27.156102,71.029741,113.185842,91.1,144.3,No312-Mg-7.6Al-0.5Zn-1Ca-2
737,12,64,52,12.126080,0.237903,12,12,73,61,68.188783,...,0.003189,52.10,1,1,23.809563,45.036179,83.845741,125.0,217.0,No383-Mg-6Al-Zn-0.3Y-0.6Gd


{'estimator': None, 'learning_rate': 0.15, 'loss': 'exponential', 'n_estimators': 50, 'random_state': None}
adaboost_tree_feature_names(top15):
Calculated Yield Strength
Grain Size
Habit Plane
frac f valence electrons
Interant d electrons
Lambda entropy
Gd fraction
Interant electrons
MagpieData mean GSvolume_pa
Radii gamma
MagpieData avg_dev Electronegativity
Calculated Grain Boundary
Zr fraction
MagpieData avg_dev MeltingT
Yang omega
average_adaboost_feature_values(top15):
0.377446570445821
0.1341845608848629
0.08848916988253257
0.03769361261344172
0.03275647789791118
0.030473669635562865
0.02447079738300413
0.018805339775549188
0.0183574964773228
0.018209693888214685
0.01495300826805707
0.012760750850656172
0.011933253664887216
0.01131924803374629
0.010921008207022275
平均 R2 分数: 0.5867462122668904
平均 MAPE 分数: 29.68279735318527
平均MAE分数: 41.874776992225804


KNN

In [25]:
from sklearn.model_selection import GridSearchCV

# 这里采用xgboost算法进行预测
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold,train_test_split,cross_validate,cross_val_score,KFold
from sklearn.metrics import r2_score, mean_absolute_percentage_error,mean_absolute_error,make_scorer
from xgboost import XGBRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import GradientBoostingRegressor  # 导入GradientBoostingRegressor
from sklearn.tree import DecisionTreeRegressor 
from sklearn.ensemble import AdaBoostRegressor
from sklearn.neighbors import KNeighborsRegressor 
from sklearn.inspection import permutation_importance
from utils import create_empty_ls,create_dir,mycopyfile,mycopydir
import joblib
from openpyxl import load_workbook
import os
"""
路径的配置和选择
"""
model_name='knn'

save_model=True
del_0qf=True #是否删除0屈服强度数据
main_path=f'models/{model_name}' # 这里是模型所在的文件位置
create_dir(main_path,is_mainpath=True)
# excel='index/xgboost_index.xlsx'
# 如果分数表格不存在就新建一个
score_path='scores_with_best_parameters.xlsx'
if os.path.exists(score_path):
    print(f"The file '{score_path}' exists.")
else:
    print(f"The file '{score_path}' does not exist.")
    df = pd.DataFrame(columns=['Model', 'R^2', 'MAPE', 'MAE'])
    df.to_excel(score_path, index=False)


"""
文件读取部分
"""
# 读取 Excel 文件
df = pd.read_excel('train_set_new.xlsx', index_col=0)  # 引入这一列之后，原本的第一列就是一个索引，第0列会从有意义的列开始
# 删除包含空值的行
if del_0qf:
    df = df[df['屈服强度'] != 0].reset_index(drop=True)
    # df = df[df['计算晶界'] != 0].reset_index(drop=True)
    # df = df[df['计算固溶'] != 0].reset_index(drop=True)
display(df)
np.random.seed(200) #设置随机数种子
feature_names = df.drop(columns=['Precipitate Distribution',
       '屈服强度', '抗拉强度 (UTS)', '追踪编号']).columns
X = df.drop(columns=['Precipitate Distribution',
       '屈服强度', '抗拉强度 (UTS)', '追踪编号'])  # 特征: 最后四列之前的所有列
y = df.iloc[:, -3]  # 目标: 倒数第四列
# 将目标变量分箱为10个类别
bins = np.linspace(y.min(), y.max(), 11)
y_binned = np.digitize(y, bins)
# print('y_binned',y_binned)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2,random_state=200)

"""
设置初始化
"""
n_splits=5
# 初始化StratifiedKFold
kf = KFold(n_splits=5, shuffle=True, random_state=200)# 注意这里修改了random_state以确保每次初始化不同

# 初始化列表以存储每个折叠的分数
# 定义评估指标
scoring = {
    'r2':make_scorer(r2_score,greater_is_better=True),
    'MAE': make_scorer(mean_absolute_error, greater_is_better=False),
    'MAPE': make_scorer(mean_absolute_percentage_error, greater_is_better=False)
}
r2_scores_ls = []
mape_scores = []
mae_scores=[]
best_r2_score = float('-inf')
best_mae_score= float('-inf')
best_mape_score= float('-inf')
ls1=['fold'+str(i) for i in range(1,n_splits+1)]
fold_dict=dict(zip(ls1,[0 for i in range(0,len(ls1))]))

# 可修改
parameters={'n_neighbors':[n for n in range(3,12,1)], # 邻居数量：3,5,7
            'algorithm':['auto','ball_tree','kd_tree','brute'], # 叶子结点所需最小样本数
            'weights':['uniform','distance'],
            'leaf_size':np.arange(25,40,5),#分裂最小样本数
            } #最大特征数量"


# 可修改
model = KNeighborsRegressor() 
# 注意这里修改了random_state以确保每次初始化不同
grid_search=GridSearchCV(model,parameters,scoring='r2',cv=kf) # 选择r2作为性能评估参数
grid_search.fit(X_train,y_train)
best_parameter=grid_search.best_estimator_


# 可修改
model = KNeighborsRegressor(n_neighbors=best_parameter.n_neighbors,algorithm=best_parameter.algorithm,weights=best_parameter.weights,leaf_size=best_parameter.leaf_size)  # 注意这里修改了random_state以确保每次初始化不同
model.fit(X_train, y_train)
# 计算分折交叉验证结果
results = cross_validate(model,X,y,cv=kf, scoring=scoring, return_train_score=False)
print(model.get_params())
# 输出分折交叉验证结果
for i, (r2, mae, mape) in enumerate(zip(results['test_r2'],results['test_MAE'], results['test_MAPE']), start=1):
    # 记录不同折的分数结果
    r2_scores_ls.append(r2)
    mape_scores.append(-mape)
    mae_scores.append(-mae)
    # print(f"折 {i}: r2={r2:.3f},MAE = {-mae:.3f}, MAPE = {-mape:.3f}")

current_r2_score=np.mean(r2_scores_ls)
# scores = cross_val_score(model, X, y_binned, cv=skf, scoring=make_scorer(r2_score))
# print('r2_scores:',scores)
# current_r2_score=scores.mean()
# 只对更优的结果进行保存
# 记录模型的特征重要性
perm_importance = permutation_importance(model, X_test, y_test, n_repeats=30, random_state=42)
# 获取特征重要性值
importances = perm_importance.importances_mean

# 对特征重要性进行排序
sorted_idx = np.argsort(importances)[::-1]

print(f'{model_name}_feature_names(top15):')
for i in feature_names[sorted_idx[:15]]:
    print(i)
print(f'average_{model_name}_feature_values(top15):')
for i in importances[sorted_idx[:15]]:
    print(i)

#记录更优的分数
best_r2_score=current_r2_score
best_mae_score=np.mean(mae_scores)
best_mape_score=np.mean(mape_scores)
if save_model:
    joblib.dump(model,   f'models/{model_name}/{model_name}_optimization1.pkl')  #保存模型
    # print(f'保存更优{model_name}模型文件') 
print('平均 R2 分数:', best_r2_score)
print('平均 MAPE 分数:', best_mape_score * 100)  # 转换为百分比
print('平均MAE分数:',best_mae_score)
"""
打印并输出分数至表格
"""

df = pd.read_excel(score_path)
# 添加新数据的示例
new_data = {'Model': f'{model_name}', 'R^2': best_r2_score, 'MAPE': best_mape_score * 100, 'MAE': best_mae_score}
new_data_df = pd.DataFrame([new_data])

# 使用 pd.concat() 合并原 DataFrame 和新的 DataFrame
df = pd.concat([df, new_data_df], ignore_index = True)
# 保存数据框到Excel表格
df.to_excel(score_path, index = False)



文件夹models/knn已存在
The file 'scores_with_best_parameters.xlsx' exists.


,MagpieData minimum Number,MagpieData maximum Number,MagpieData range Number,MagpieData mean Number,MagpieData avg_dev Number,MagpieData mode Number,MagpieData minimum MendeleevNumber,MagpieData maximum MendeleevNumber,MagpieData range MendeleevNumber,MagpieData mean MendeleevNumber,...,frac f valence electrons,Grain Size,Precipitate Distribution,Habit Plane,Calculated Solid Solution,Calculated Grain Boundary,Calculated Yield Strength,屈服强度,抗拉强度 (UTS),追踪编号
0,12,30,18,12.131276,0.247009,12,52,73,21,68.266826,...,0.000000,4.00,0,0,17.669989,194.321983,226.991971,364.0,393.0,No630-6Al-1Zn-0.15Mn -3
1,12,64,52,12.922815,1.811506,12,23,68,45,67.259700,...,0.054984,36.00,3,1,57.602249,48.504397,121.106646,125.0,200.0,No232-Mg-9Gd-1Sm-0.5Zr-2
2,12,64,52,13.780587,3.401587,12,12,69,57,66.376359,...,0.078055,50.00,3,5,94.613373,69.789053,179.402426,200.0,340.0,No16-Mg–14Gd–3Y–1.8Zn–0.5Zr -9
3,12,64,52,13.179217,2.292508,12,12,68,56,66.814372,...,0.052520,1.50,2,1,76.138523,262.520047,353.658570,290.0,280.0,No96-Mg-9.3Gd-2.7Y-0.65Ag-0.52Zr-0.18Ce -4
4,12,30,18,12.112392,0.217496,12,52,73,21,68.118900,...,0.000000,15.00,0,0,11.604694,86.484728,113.089422,148.0,280.0,No460-3Al-1Zn-0.3Mn -22
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
734,12,38,26,12.053936,0.104733,12,8,73,65,68.099036,...,0.000000,130.00,1,1,12.020723,23.509472,50.530195,58.5,220.0,No744-Mg-3Al-0.4Mn-0.05Sr
735,12,64,52,12.937303,1.825661,12,12,69,57,67.051202,...,0.033095,41.77,3,5,61.765688,53.715298,130.480986,223.0,295.0,No226-Mg-6Gd-3Y-1Zn-0.4Zr-0.7Ag-3
736,12,30,18,12.152650,0.281635,12,7,73,66,67.973850,...,0.000000,30.00,3,2,27.156102,71.029741,113.185842,91.1,144.3,No312-Mg-7.6Al-0.5Zn-1Ca-2
737,12,64,52,12.126080,0.237903,12,12,73,61,68.188783,...,0.003189,52.10,1,1,23.809563,45.036179,83.845741,125.0,217.0,No383-Mg-6Al-Zn-0.3Y-0.6Gd


{'algorithm': 'ball_tree', 'leaf_size': 30, 'metric': 'minkowski', 'metric_params': None, 'n_jobs': None, 'n_neighbors': 8, 'p': 2, 'weights': 'distance'}
knn_feature_names(top15):
Calculated Yield Strength
Calculated Grain Boundary
Grain Size
Calculated Solid Solution
MagpieData maximum AtomicWeight
MagpieData range AtomicWeight
MagpieData range CovalentRadius
MagpieData maximum SpaceGroupNumber
MagpieData maximum Number
MagpieData range SpaceGroupNumber
range Number
MagpieData range Number
MagpieData minimum MendeleevNumber
Habit Plane
MagpieData maximum CovalentRadius
average_knn_feature_values(top15):
0.47750724170806075
0.20691454257676736
0.1508817986828809
0.03634377512131594
0.029901995831987434
0.029375792200559008
0.02414920306481925
0.023061403759098275
0.014677072076202632
0.014414007528938354
0.014329446959827996
0.014329446959827992
0.011071256916646307
0.009753438371125937
0.00961680731618358
平均 R2 分数: 0.5948627946387235
平均 MAPE 分数: 25.156123593572744
平均MAE分数: 38.9309759

In [23]:
df = pd.read_excel(score_path)
# 添加新数据的示例
new_data = {'Model': f'{model_name}', 'R^2': best_r2_score, 'MAPE': best_mape_score * 100, 'MAE': best_mae_score}
new_data_df = pd.DataFrame([new_data])

# 使用 pd.concat() 合并原 DataFrame 和新的 DataFrame
df = pd.concat([df, new_data_df], ignore_index=True)
# 保存数据框到Excel表格
df.to_excel(score_path, index = False)


In [24]:
model = KNeighborsRegressor(n_neighbors=5)  # 注意这里修改了random_state以确保每次初始化不同
model.fit(X_train, y_train)
# 计算分折交叉验证结果
results = cross_validate(model,X,y,cv=kf, scoring=scoring, return_train_score=False)
# 输出分折交叉验证结果
for i, (r2, mae, mape) in enumerate(zip(results['test_r2'],results['test_MAE'], results['test_MAPE']), start=1):
    # 记录不同折的分数结果
    r2_scores_ls.append(r2)
    mape_scores.append(-mape)
    mae_scores.append(-mae)
    # print(f"折 {i}: r2={r2:.3f},MAE = {-mae:.3f}, MAPE = {-mape:.3f}")

current_r2_score=np.mean(r2_scores_ls)
# scores = cross_val_score(model, X, y_binned, cv=skf, scoring=make_scorer(r2_score))
# print('r2_scores:',scores)
# current_r2_score=scores.mean()
# 只对更优的结果进行保存
# 记录模型的特征重要性
perm_importance = permutation_importance(model, X_test, y_test, n_repeats=30, random_state=42)
        # 获取特征重要性值
importances = perm_importance.importances_mean

# 对特征重要性进行排序
sorted_idx = np.argsort(importances)[::-1]

print(f'{model_name}_feature_names(top15):')
for i in feature_names[sorted_idx[:15]]:
    print(i)
print(f'average_{model_name}_feature_values(top15):')
for i in importances[sorted_idx[:15]]:
    print(i)
#记录更优的分数
best_r2_score=current_r2_score
best_mae_score=np.mean(mae_scores)
best_mape_score=np.mean(mape_scores)

print('平均 R2 分数:', best_r2_score)
print('平均 MAPE 分数:', best_mape_score * 100)  # 转换为百分比
print('平均MAE分数:',best_mae_score)
model.get_params()

knn_feature_names(top15):
Calculated Yield Strength
Calculated Grain Boundary
Grain Size
Calculated Solid Solution
MagpieData range SpaceGroupNumber
MagpieData maximum SpaceGroupNumber
MagpieData range CovalentRadius
MagpieData range AtomicWeight
MagpieData maximum AtomicWeight
MagpieData maximum Number
MagpieData range Number
range Number
Habit Plane
MagpieData range MendeleevNumber
MagpieData avg_dev SpaceGroupNumber
average_knn_feature_values(top15):
0.5373967222432158
0.23889804944175666
0.1743922322775962
0.05653516910396467
0.02399062196353012
0.017401812603493307
0.012413300038396417
0.008626908999037715
0.008515076648588337
0.004849838622909474
0.004849795003155732
0.004849795003155732
0.002954841392850834
0.0014303485914000842
0.00042025292185045765
平均 R2 分数: 0.5755978719306099
平均 MAPE 分数: 25.993147341427026
平均MAE分数: 40.1941662219052


{'algorithm': 'auto',
 'leaf_size': 30,
 'metric': 'minkowski',
 'metric_params': None,
 'n_jobs': None,
 'n_neighbors': 5,
 'p': 2,
 'weights': 'uniform'}

In [26]:
from sklearn.neighbors import KNeighborsRegressor
from sklearn.model_selection import cross_validate, KFold
from sklearn.inspection import permutation_importance
import numpy as np

# 定义参数
params = {'algorithm': 'ball_tree', 'leaf_size': 30, 'metric': 'minkowski', 'metric_params': None, 'n_jobs': None, 'p': 2, 'weights': 'distance'}

# 定义交叉验证
kf = KFold(n_splits=5, shuffle=True, random_state=200)

# 定义评分指标
scoring = ['r2', 'neg_mean_absolute_error', 'neg_mean_absolute_percentage_error']
df = pd.read_excel('train_set_new.xlsx', index_col=0)  # 引入这一列之后，原本的第一列就是一个索引，第0列会从有意义的列开始

# 定义特征和目标变量
X = df.drop(columns=['Precipitate Distribution',
      '屈服强度', '抗拉强度 (UTS)', '追踪编号'])  # 特征: 最后四列之前的所有列
y = df.iloc[:, -3]  # 目标: 倒数第四列
# 初始化记录结果的变量
best_r2_score = float('-inf')
best_n_neighbors = None
r2_scores_ls = []
mape_scores = []
mae_scores = []

# 循环不同的n_neighbors值
for n_neighbors in range(3, 13):
    model = KNeighborsRegressor(n_neighbors=n_neighbors, **params)
    model.fit(X_train, y_train)

    # 计算分折交叉验证结果
    results = cross_validate(model, X, y, cv=kf, scoring=scoring, return_train_score=False)

    # 输出分折交叉验证结果
    for i, (r2, mae, mape) in enumerate(zip(results['test_r2'], results['test_neg_mean_absolute_error'], results['test_neg_mean_absolute_percentage_error']), start=1):
        # 记录不同折的分数结果
        r2_scores_ls.append(r2)
        mape_scores.append(-mape)
        mae_scores.append(-mae)
    
    current_r2_score = np.mean(r2_scores_ls)
    
    if current_r2_score > best_r2_score:
        best_r2_score = current_r2_score
        best_mae_score = np.mean(mae_scores)
        best_mape_score = np.mean(mape_scores)
        best_n_neighbors = n_neighbors

        # 记录模型的特征重要性
        perm_importance = permutation_importance(model, X_test, y_test, n_repeats=30, random_state=42)
        importances = perm_importance.importances_mean
        sorted_idx = np.argsort(importances)[::-1]

        print(f'KNeighborsRegressor with n_neighbors={n_neighbors} feature importance (top 15):')
        for i in feature_names[sorted_idx[:15]]:
            print(i)
        print(f'Average feature importances (top 15):')
        for i in importances[sorted_idx[:15]]:
            print(i)

print('最佳 n_neighbors:', best_n_neighbors)
print('平均 R2 分数:', best_r2_score)
print('平均 MAPE 分数:', best_mape_score * 100)  # 转换为百分比
print('平均 MAE 分数:', best_mae_score)


KNeighborsRegressor with n_neighbors=3 feature importance (top 15):
Calculated Yield Strength
Calculated Grain Boundary
Grain Size
Calculated Solid Solution
Habit Plane
MagpieData range CovalentRadius
MagpieData maximum AtomicWeight
MagpieData range AtomicWeight
MagpieData maximum CovalentRadius
MagpieData range NsUnfilled
Interant s electrons
MagpieData maximum NsUnfilled
MagpieData range NsValence
MagpieData minimum NsValence
MagpieData minimum GSmagmom
Average feature importances (top 15):
0.5906431931055126
0.23117018259018654
0.14731407472789115
0.04627343769589602
0.035993804117607095
0.029748538961130267
0.009486872836393555
0.00877559542831512
0.0015790973904999432
0.0002935806872154186
0.0002935806872154186
0.0002935806872154186
0.0002935806872154112
0.0002935806872154112
0.0
KNeighborsRegressor with n_neighbors=4 feature importance (top 15):
Calculated Yield Strength
Calculated Grain Boundary
Grain Size
Calculated Solid Solution
Habit Plane
MagpieData range CovalentRadius
Mag